# Análisis de documentos y evidencias

Extracción de señales de documentos Word y construcción de relaciones entre archivos.

Rutas del entorno original: /home/jupyter/...

# Imports

In [ ]:
import pandas as pd
import zipfile
import os
import re
import xml.etree.ElementTree as ET
import olefile
import struct
from urllib.parse import urlparse, unquote
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
import math
import networkx as nx
import matplotlib.colors as mcolors
from pyvis.network import Network
from datetime import datetime
import io
import hashlib
from pathlib import Path
from PIL import Image
from io import BytesIO
import imagehash
from PIL.ExifTags import TAGS
import itertools

# Funciones Basicas

Funciones de identificación de formato y carga inicial del conjunto de datos.

In [ ]:
def check_ole2(path):
    siganture = b'\xD0\xCF\x11\xE0\xA1\xB1\x1A\xE1'
    try:
        with open(path, 'rb') as f:
            header = f.read(8) # leemos los primeros 8 bytes del archivo
            return header == siganture
    except:
        return False
    
def check_ooxml(path):
    siganture = b'PK\x03\x04'

    try:
        with open(path, 'rb') as f:
            header = f.read(4) # leemos los primeros 4 bytes del archivo
            return header == siganture
    except:
        return False
    
def check_rtf(path):
    signature = b'{\\rtf' 
    try:
        with open(path, 'rb') as f:
            header = f.read(4) # leemos 4 bytes
            return header == signature
    except:
        return False
    
def check_pdf(path):
    signature = b'%PDF'
    try:
        with open(path, 'rb') as f:
            header = f.read(4) # leemos 4 bytes
            return header == signature
    except:
        return False
    
def check_exe(path):
    # Los ejecutables de Windows (.exe, .dll) empiezan por 'MZ'
    signature = b'MZ'
    try:
        with open(path, 'rb') as f:
            header = f.read(2) # leemos solo 2 bytes
            return header == signature
    except:
        return False
    

def check_file(path):
    if check_ole2(path):
        return 'ole2'
    elif check_ooxml(path):
        return 'ooxml'
    elif check_rtf(path):
        return 'rtf'
    elif check_pdf(path):
        return 'pdf'
    elif check_exe(path):
        return 'exe'
    else:
        return 'unknown'


def analyze_distribution(counter):
    panose_distribution = defaultdict(int)
    for panose, count in counter.items():
        panose_distribution[count] += 1

    # hacemos un analisis de la distribucion, con percentiles, media, mediana, moda , etc...
    # ANÁLISIS DE LA DISTRIBUCIÓN (ESTADÍSTICO)

    # 1. Extraemos las frecuencias (los "counts") a una Serie de Pandas
    frecuencias = list(counter.values())
    frecuencias_series = pd.Series(frecuencias)

    print("\n=== ANÁLISIS DE DISTRIBUCIÓN DE LAS FRECUENCIAS DE LA CARACTERÍSTICA ===")

    # 2. Calculamos media, mediana y percentiles usando describe()
    # Le pasamos una lista de percentiles para tener una visión más detallada
    estadisticas = frecuencias_series.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

    print("\n--- Estadísticas Descriptivas y Percentiles ---")
    print(estadisticas.round(2)) # Redondeamos a 2 decimales para que sea más legible

    # 3. Calculamos la Moda (el número de apariciones que más se repite)
    # (mode() devuelve una serie porque puede ser bimodal/multimodal, lo pasamos a lista)
    moda = frecuencias_series.mode().tolist()
    print("\n--- Medidas de Tendencia Central y Dispersión ---")
    print(f"Moda: {moda}")

    # 4. Otras métricas útiles: Varianza, Asimetría (Skewness) y Curtosis
    varianza = frecuencias_series.var()
    asimetria = frecuencias_series.skew()
    curtosis = frecuencias_series.kurt()

    print(f"Varianza: {varianza:.2f}")
    print(f"Asimetría (Skewness): {asimetria:.2f} (Si es > 0, tiene cola hacia la derecha)")
    print(f"Curtosis: {curtosis:.2f} (Pico de la distribución)")

    # 5. Para dar contexto, mostramos los 10 valores más frecuentes
    print("\n--- Top 10 valores más presentes en el corpus ---")
    for caracteristica, count in counter.most_common(10):
        print(f"Valor: {caracteristica}  ->  {count} apariciones")

    # VISUALIZACIÓN BÁSICA
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 5))
    # Histograma de las frecuencias, usando escala logarítmica si hay mucha diferencia
    plt.hist(frecuencias, bins=50, color='skyblue', edgecolor='black', log=True)
    plt.title('Distribución de frecuencias de los valores de la característica')
    plt.xlabel('Número de apariciones en el dataset')
    plt.ylabel('Cantidad de valores únicos')
    plt.show()

In [ ]:
csv_full = pd.read_csv('/home/jupyter/ADAPTDataset.csv')

# PANOSE y Fuentes

Extracción y comparación de identificadores de fuentes en documentos OOXML y OLE2.

## OOXML

In [ ]:
def search_PANOSE_USB_CSB(ooxml_path, verbose=False):
    try:
        with zipfile.ZipFile(ooxml_path, 'r') as zip_ref:

            if 'word/fontTable.xml' in zip_ref.namelist():
                contenido_document = zip_ref.read('word/fontTable.xml')
                #print(contenido_document.decode('utf-8'))

               
                root = ET.fromstring(contenido_document.decode('utf-8'))
            
                # sacar las fuentes usadas en el documento, con su nombre y su charset
                # intentar sacar el w:usbX y w:csbX
                ns_w = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'

                returned_data = []


                for font in root.iter(f'{ns_w}font'):
                    # Sacar el nombre de la fuente
                    name = font.attrib.get(f'{ns_w}name', 'Desconocido')
                    
                    # Sacar el charset
                    charset = font.find(f'{ns_w}charset')
                    charset_val = charset.attrib.get(f'{ns_w}val') if charset is not None else 'N/A'
                    
                    # Sacar el panose1
                    panose = font.find(f'{ns_w}panose1')
                    panose_val = panose.attrib.get(f'{ns_w}val') if panose is not None else 'N/A'
                    
                    # Sacar los valores usbX y csbX de la etiqueta <w:sig>
                    sig = font.find(f'{ns_w}sig')
                    if sig is not None:
                        usb0 = sig.attrib.get(f'{ns_w}usb0', 'N/A')
                        usb1 = sig.attrib.get(f'{ns_w}usb1', 'N/A')
                        usb2 = sig.attrib.get(f'{ns_w}usb2', 'N/A')
                        usb3 = sig.attrib.get(f'{ns_w}usb3', 'N/A')
                        csb0 = sig.attrib.get(f'{ns_w}csb0', 'N/A')
                        csb1 = sig.attrib.get(f'{ns_w}csb1', 'N/A')
                    else:
                        # Algunas fuentes (raro, pero posible) podri­an no tener la firma declarada
                        usb0 = usb1 = usb2 = usb3 = csb0 = csb1 = 'N/A'
                        
                    if verbose:
                        print(f"- Fuente: {name}")
                        print(f"  Charset: {charset_val} | Panose: {panose_val}")
                        print(f"  USB: {usb0}-{usb1}-{usb2}-{usb3}")
                        print(f"  CSB: {csb0}-{csb1}")
                        print("-" * 40)

                    returned_data.append({
                        "sha256": ooxml_path.split('/')[-1],
                        "name": name,
                        "charset": charset_val,
                        "panose": panose_val,
                        "usb": f"{usb0}-{usb1}-{usb2}-{usb3}",
                        "csb": f"{csb0}-{csb1}"
                    })

                return returned_data
            
    except Exception as e:
        print(f"Error procesando el zip de {ooxml_path}: {e}")


# diccionario que se va a transformar en un df con columnas: sha256, file_type, panose, USB, CSB
font_data = []

for n, docname in csv_full['sha256'].items():
    if check_ooxml("/home/jupyter/binaries/" + docname):
        #print(f"Analizando PANOSE, USB y CSB en: {docname}")
        font_info = search_PANOSE_USB_CSB("/home/jupyter/binaries/" + docname, verbose=True)
        if font_info:
            font_data.extend(font_info)





In [ ]:
font_data_df = pd.DataFrame(font_data)
# ordenarlo por sha256
font_data_ooxml = font_data_df.sort_values(by='sha256').reset_index(drop=True)
font_data_ooxml

In [ ]:
# ver cuantos valores unicos hay de sha256 y compararlos con el numero de ooxml en csv_full
unique_sha256_count = font_data_ooxml['sha256'].nunique()

print(f"Valores unicos de sha256 en font_data_ooxml: {unique_sha256_count}")

total = 0
total_no_font_data = 0

for docname in csv_full['sha256'].items():
    if check_ooxml("/home/jupyter/binaries/" + docname[1]):
        total += 1
        if docname[1] not in font_data_ooxml['sha256'].values:
            #print(f"El documento {docname[1]} es OOXML pero no tiene datos de fuente en font_data_ooxml.")
            total_no_font_data += 1

print(f"Total de documentos OOXML: {total}")
print(f"Total de documentos OOXML sin datos de fuente: {total_no_font_data}")

In [ ]:
# 1. Filtramos el DataFrame para quedarnos solo con las filas que tienen ese panose
fuentes_con_panose_0 = font_data_ooxml[font_data_ooxml['panose'] == '00000000000000000000']

# 2. Contamos cuantas veces aparece cada nombre de fuente en ese grupo filtrado
panose_counter = fuentes_con_panose_0['name'].value_counts()

# Mostrar el resultado
print(panose_counter)

In [ ]:
# Contar las combinaciones de usb (o csb cambiando 'usb' por 'csb' en estas lineas) y name en todo el dataset
conteo_grupo = font_data_ooxml[['usb', 'name']].value_counts().reset_index(name='cantidad')

# unir el usb y name con la cantidad de veces que aparecen juntos y meterlos en un counter
panose_counter = Counter(zip(font_data_ooxml['usb'], font_data_ooxml['name']))

# ahora ponemos un diccionario donde la llave es el numero de apariciones y el valor es el numero de codigos panose con ese numero de apariciones
panose_distribution = defaultdict(int)
for panose, count in panose_counter.items():
    panose_distribution[count] += 1

# hacemos un analisis de la distribucion, con percentiles, media, mediana, moda , etc...

# 1. Extraemos las frecuencias (los "counts") a una Serie de Pandas
frecuencias = list(panose_counter.values())
frecuencias_series = pd.Series(frecuencias)

print("\n=== ANÁLISIS DE DISTRIBUCIÓN DE LAS FRECUENCIAS DE PANOSE ===")

# 2. Calculamos media, mediana y percentiles usando describe()
# Le pasamos una lista de percentiles para tener una visión más detallada
estadisticas = frecuencias_series.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

print("\n--- Estadísticas Descriptivas y Percentiles ---")
print(estadisticas.round(2)) # Redondeamos a 2 decimales para que sea más legible

# 3. Calculamos la Moda (el número de apariciones que más se repite)
# (mode() devuelve una serie porque puede ser bimodal/multimodal, lo pasamos a lista)
moda = frecuencias_series.mode().tolist()
print("\n--- Medidas de Tendencia Central y Dispersión ---")
print(f"Moda: {moda}")

# 4. Otras métricas útiles: Varianza, Asimetría (Skewness) y Curtosis
varianza = frecuencias_series.var()
asimetria = frecuencias_series.skew()
curtosis = frecuencias_series.kurt()

print(f"Varianza: {varianza:.2f}")
print(f"Asimetria (Skewness): {asimetria:.2f} (Si es > 0, tiene cola hacia la derecha)")
print(f"Curtosis: {curtosis:.2f} (Pico de la distribución)")

# 5. Para dar contexto, mostramos los 10 códigos PANOSE reales más frecuentes
print("\n--- Top 10 Códigos PANOSE más presentes en el corpus ---")
for panose, count in panose_counter.most_common(10):
    print(f"PANOSE: {panose}  ->  {count} apariciones")

# VISUALIZACIÓN BÁSICA
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
# Histograma de las frecuencias, usando escala logarítmica si hay mucha diferencia
plt.hist(frecuencias, bins=50, color='skyblue', edgecolor='black', log=True)
plt.title('Distribución de frecuencias de los Códigos PANOSE (Escala Logarítmica)')
plt.xlabel('Numero de apariciones en el dataset')
plt.ylabel('Cantidad de codigos PANOSE unicos')
plt.show()

In [ ]:
# Contar las combinaciones de panose y name en todo el dataset
conteo_grupo = font_data_ooxml[['panose', 'name']].value_counts().reset_index(name='cantidad')

# unir el panose y name con la cantidad de veces que aparecen juntos y meterlos en un counter
panose_counter = Counter(zip(font_data_ooxml['panose'], font_data_ooxml['name']))

# ahora ponemos un diccionario donde la llave es el numero de apariciones y el valor es el numero de codigos panose con ese numero de apariciones
panose_distribution = defaultdict(int)
for panose, count in panose_counter.items():
    panose_distribution[count] += 1

# hacemos un analisis de la distribucion, con percentiles, media, mediana, moda , etc...
# ANÁLISIS DE LA DISTRIBUCIÓN (ESTADÍSTICO)

# 1. Extraemos las frecuencias (los "counts") a una Serie de Pandas
frecuencias = list(panose_counter.values())
frecuencias_series = pd.Series(frecuencias)

print("\n=== ANÁLISIS DE DISTRIBUCIÓN DE LAS FRECUENCIAS DE PANOSE ===")

# 2. Calculamos media, mediana y percentiles usando describe()
# Le pasamos una lista de percentiles para tener una visión más detallada
estadisticas = frecuencias_series.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

print("\n--- Estadísticas Descriptivas y Percentiles ---")
print(estadisticas.round(2)) # Redondeamos a 2 decimales para que sea más legible

# 3. Calculamos la Moda (el número de apariciones que más se repite)
# (mode() devuelve una serie porque puede ser bimodal/multimodal, lo pasamos a lista)
moda = frecuencias_series.mode().tolist()
print("\n--- Medidas de Tendencia Central y Dispersión ---")
print(f"Moda: {moda}")

# 4. Otras métricas útiles: Varianza, Asimetría (Skewness) y Curtosis
varianza = frecuencias_series.var()
asimetria = frecuencias_series.skew()
curtosis = frecuencias_series.kurt()

print(f"Varianza: {varianza:.2f}")
print(f"Asimetría (Skewness): {asimetria:.2f} (Si es > 0, tiene cola hacia la derecha)")
print(f"Curtosis: {curtosis:.2f} (Pico de la distribución)")

# 5. Para dar contexto, mostramos los 10 códigos PANOSE reales más frecuentes
print("\n--- Top 10 Códigos PANOSE más presentes en el corpus ---")
for panose, count in panose_counter.most_common(10):
    print(f"PANOSE: {panose}  ->  {count} apariciones")

# VISUALIZACIÓN BÁSICA
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
# Histograma de las frecuencias, usando escala logarítmica si hay mucha diferencia
plt.hist(frecuencias, bins=50, color='skyblue', edgecolor='black', log=True)
plt.title('Distribución de frecuencias de los Códigos PANOSE (Escala Logarítmica)')
plt.xlabel('Número de apariciones en el dataset')
plt.ylabel('Cantidad de códigos PANOSE únicos')
plt.show()

In [ ]:
# Filtrado de name+panose por frecuencia

upper_limit = 20

filtered_panose_counter = {k: v for k, v in panose_counter.items() if v > 1 and v < upper_limit}  # Límite superior

filtered_panose_counter

!!! Esta con USB o CSB no con PANOSE

In [ ]:
# Parejas de documentos con name+panose compartido
docs_parejas = []

for i, row1 in font_data_ooxml.iterrows():
    for j, row2 in font_data_ooxml.iterrows():
        if i < j:  # Evitar duplicados y auto-comparación
            if (row1['usb'], row1['name']) == (row2['usb'], row2['name']):

                # solo agregamos la pareja si el name+panose está en el filtered_panose_counter
                if (row1['usb'], row1['name']) in filtered_panose_counter:
                    docs_parejas.append((row1['sha256'], row2['sha256'], row1['usb'], row1['name']))

# Coincidencias de name+panose por pareja de documentos
docs_parejas_counter = Counter((doc1, doc2) for doc1, doc2, _, _ in docs_parejas)
docs_parejas_counter

In [ ]:
# a partir de docs_parejas_counter, creamos un grafo donde los nodos son los documentos y las aristas representan que comparten al menos un name+panose, 
# con el peso de la arista siendo el número de name+panose compartidos

G_usb = nx.Graph()
for (doc1, doc2), weight in docs_parejas_counter.items():
    G_usb.add_edge(doc1, doc2, weight=weight)

# 4. Asignar atributos optimizado con diccionario
sha_to_apt = csv_full.set_index('sha256')['Normalized_Tag'].to_dict()

for node in G_usb.nodes():
    # Formato
    ruta = f"/home/jupyter/binaries/{node}"
    G_usb.nodes[node]['format'] = 'OOXML' if check_ooxml(ruta) else 'Otro'
    
    # APT
    G_usb.nodes[node]['apt'] = sha_to_apt.get(node, 'Desconocido')

# 5. Estadísticas
num_nodos = G_usb.number_of_nodes()
num_aristas = G_usb.number_of_edges()
print(f"El grafo tiene {num_nodos} nodos y {num_aristas} aristas.")

In [ ]:
def merge_graphs_weighted(graphs: list[nx.Graph], 
                           graph_weights: list[float],
                           graph_names: list[str] = None) -> nx.Graph:
    """
    Combina grafos aplicando un multiplicador a cada uno antes de sumar.
    
    Ejemplo de pesos sugeridos para tu caso:
      - USB:    0.5  (muy específico, alta confianza)
      - CSB:    0.3  (específico pero más común)
      - Panose: 0.2  (más genérico)
    """
    assert len(graphs) == len(graph_weights), "Un peso por grafo"
    
    G_merged = nx.Graph()
    
    for i, (G, gw) in enumerate(zip(graphs, graph_weights)):
        graph_id = graph_names[i] if graph_names else f"graph_{i}"
        
        for u, v, data in G.edges(data=True):
            w = data.get('weight', 1) * gw  # peso ponderado
            
            if G_merged.has_edge(u, v):
                G_merged[u][v]['weight'] += w
                G_merged[u][v]['sources'].append(graph_id)
                G_merged[u][v]['raw_weights'][graph_id] = data.get('weight', 1)
            else:
                G_merged.add_edge(u, v,
                                  weight=w,
                                  sources=[graph_id],
                                  raw_weights={graph_id: data.get('weight', 1)})
    
    return G_merged




def normalize_graph_weights(G: nx.Graph) -> nx.Graph:
    """Normaliza pesos al rango [0, 1]"""
    G_norm = G.copy()
    weights = [d['weight'] for _, _, d in G.edges(data=True)]
    
    if not weights:
        return G_norm
    
    w_min, w_max = min(weights), max(weights)
    
    if w_max == w_min:
        # Todos iguales → peso uniforme
        for u, v in G_norm.edges():
            G_norm[u][v]['weight'] = 1.0
    else:
        for u, v in G_norm.edges():
            w = G_norm[u][v]['weight']
            G_norm[u][v]['weight'] = (w - w_min) / (w_max - w_min)
    
    return G_norm

# Pipeline completo recomendado
G_panose_n = normalize_graph_weights(G_panose)
G_usb_n    = normalize_graph_weights(G_usb)
G_csb_n    = normalize_graph_weights(G_csb)

G_final = merge_graphs_weighted(
    graphs=[G_panose_n, G_usb_n, G_csb_n],
    graph_weights=[0.2, 0.5, 0.3],
    graph_names=['panose', 'usb', 'csb']
)

In [ ]:
# Hacemos una copia para exportar a Gephi
G_export = G_final.copy()

for u, v, data in G_export.edges(data=True):
    # Convertimos la lista en un texto separado por comas: "panose, usb"
    if 'sources' in data:
        data['sources'] = ", ".join(data['sources'])
    
    # Convertimos el diccionario a texto: "{'panose': 0.8, 'usb': 1.0}"
    if 'raw_weights' in data:
        data['raw_weights'] = str(data['raw_weights'])

# Ahora exportará sin problemas
nx.write_gexf(G_export, "grafo_final_fonts_ponderado_ooxml.gexf")

In [ ]:
# exportar para gephi
# nx.write_gexf(G_panose, "grafo_panose_ooxml.gexf")
# nx.write_gexf(G_usb, "grafo_usb_ooxml.gexf")
# nx.write_gexf(G_csb, "agrafo_csb_ooxml.gexf")
# nx.write_gexf(G_final, "grafo_final_fonts_ponderado.gexf")

## OLE2

In [ ]:
def extraer_fuentes_ole2(doc_path, verbose=False):
    returned_data = []

    try:
        with olefile.OleFileIO(doc_path) as ole:
            if not ole.exists('WordDocument'):
                return []
            wordstream = ole.openstream('WordDocument').read()

            flag = struct.unpack("<H", wordstream[10:12])[0]
            table_name = "1Table" if (flag & 0x200) else "0Table"
            if not ole.exists(table_name):
                return []
            tablestream = ole.openstream(table_name).read()

            fcSttbfffn  = struct.unpack("<I", wordstream[274:278])[0]
            lcbSttbfffn = struct.unpack("<I", wordstream[278:282])[0]

            if fcSttbfffn == 0 or lcbSttbfffn == 0:
                return []

            sttbf_data = tablestream[fcSttbfffn : fcSttbfffn + lcbSttbfffn]
            if len(sttbf_data) < 4:
                return []

            cData   = struct.unpack("<H", sttbf_data[0:2])[0]
            cbExtra = struct.unpack("<H", sttbf_data[2:4])[0]
            offset  = 4

            for _ in range(cData):
                if offset >= len(sttbf_data):
                    break

                cbFfnM1  = sttbf_data[offset]
                ffn_data = sttbf_data[offset : offset + cbFfnM1 + 1]
                offset  += cbFfnM1 + 1
                offset  += cbExtra

                if len(ffn_data) < 41:
                    continue

                panose_hex = ffn_data[6:16].hex().upper()
                if panose_hex == '00000000000000000000':
                    panose_hex = 'N/A'

                fs_data = ffn_data[16:40]
                usb = struct.unpack("<IIII", fs_data[0:16])
                csb = struct.unpack("<II",   fs_data[16:24])

                prg_byte    = ffn_data[1]
                is_truetype = bool(prg_byte & 0x04)
                pitch       = prg_byte & 0x03
                family      = (prg_byte >> 4) & 0x0F

                charset = f"{ffn_data[4]:02X}"
                weight  = struct.unpack("<H", ffn_data[2:4])[0]

                name_raw = ffn_data[40:]
                for i in range(0, len(name_raw) - 1, 2):
                    if name_raw[i] == 0 and name_raw[i+1] == 0:
                        name_raw = name_raw[:i]
                        break
                font_name = name_raw.decode('utf-16le', errors='ignore').strip()

                if not font_name:
                    continue

                if verbose:
                    print(f"- Fuente: {font_name}")
                    print(f"  Charset: {charset} | Weight: {weight} | TrueType: {is_truetype}")
                    print(f"  Panose: {panose_hex}")
                    print(f"  USB: {usb[0]:08X}-{usb[1]:08X}-{usb[2]:08X}-{usb[3]:08X}")
                    print(f"  CSB: {csb[0]:08X}-{csb[1]:08X}")
                    print("-" * 40)

                returned_data.append({
                    'sha256':       doc_path.split('/')[-1],
                    'name':         font_name,
                    'charset':      charset,
                    'panose':       panose_hex,
                    'usb':          f"{usb[0]:08X}-{usb[1]:08X}-{usb[2]:08X}-{usb[3]:08X}",
                    'csb':          f"{csb[0]:08X}-{csb[1]:08X}",
                    'weight':       weight,
                    'is_truetype':  is_truetype,
                    'family':       family,
                    'pitch':        pitch,
                })

    except Exception as e:
        print(f"Error parseando fuentes OLE2 en {doc_path}: {e}")

    return returned_data


font_data = []

for n, docname in csv_full['sha256'].items():    
    if check_ole2("/home/jupyter/binaries/" + docname):
        #print(f"Analizando fuentes en OLE2: {docname}")
        fuentes = extraer_fuentes_ole2("/home/jupyter/binaries/" + docname)
        if fuentes:
            font_data.extend(fuentes)



In [ ]:
import zipfile

def destripar_metadatos_doc2(ruta_archivo):
    # Abrimos el docx como lo que es: un simple ZIP
    with zipfile.ZipFile(ruta_archivo, 'r') as docx_zip:
        try:
            # Leemos el archivo core.xml directamente
            core_xml_bruto = docx_zip.read('docProps/core.xml')
            print("--- CONTENIDO BRUTO DE METADATOS DOC2 ---")
            print(core_xml_bruto.decode('utf-8'))
        except KeyError:
            print("El archivo docProps/core.xml ni siquiera existe.")

# Pon aquí la ruta de tu doc2
destripar_metadatos_doc2("/home/jupyter/workspace/doc2.docx")

In [ ]:
font_ole_df = pd.DataFrame(font_data)
font_data_ole2 = font_ole_df.sort_values(by='sha256').reset_index(drop=True)
font_data_ole2

In [ ]:
# ver cuantos valores únicos hay de sha256 y compararlos con el numero de ole2 en csv_full
unique_sha256_count = font_data_ole2['sha256'].nunique()

print(f"Valores únicos de sha256 en font_data_ole2: {unique_sha256_count}")

total = 0
total_no_font_data = 0

for docname in csv_full['sha256'].items():
    if check_ole2("/home/jupyter/binaries/" + docname[1]):
        total += 1
        if docname[1] not in font_data_ole2['sha256'].values:
            #print(f"El documento {docname[1]} es OOXML pero no tiene datos de fuente en font_data_ole2.")
            total_no_font_data += 1

print(f"Total de documentos OOXML: {total}")
print(f"Total de documentos OOXML sin datos de fuente: {total_no_font_data}")

In [ ]:
# 1. Filtramos el DataFrame para quedarnos solo con las filas que tienen ese panose
# fuentes_con_panose_0 = font_data_ole2[font_data_ole2['panose'] == '00000000000000000000']
fuentes_con_panose_0 = font_data_ole2[font_data_ole2['panose'] == 'N/A']

# 2. Contamos cuántas veces aparece cada nombre de fuente en ese grupo filtrado
panose_counter = fuentes_con_panose_0['name'].value_counts()

# Mostrar el resultado
print(panose_counter)

In [ ]:
# Contar las combinaciones de panose y name en todo el dataset
conteo_grupo = font_data_ole2[['panose', 'name']].value_counts().reset_index(name='cantidad')

# unir el panose y name con la cantidad de veces que aparecen juntos y meterlos en un counter
panose_counter = Counter(zip(font_data_ole2['panose'], font_data_ole2['name']))

# ahora ponemos un diccionario donde la llave es el numero de apariciones y el valor es el numero de codigos panose con ese numero de apariciones
panose_distribution = defaultdict(int)
for panose, count in panose_counter.items():
    panose_distribution[count] += 1

# hacemos un analisis de la distribucion, con percentiles, media, mediana, moda , etc...
# ANÁLISIS DE LA DISTRIBUCIÓN (ESTADÍSTICO)

# 1. Extraemos las frecuencias (los "counts") a una Serie de Pandas
frecuencias = list(panose_counter.values())
frecuencias_series = pd.Series(frecuencias)

print("\n=== ANÁLISIS DE DISTRIBUCIÓN DE LAS FRECUENCIAS DE PANOSE ===")

# 2. Calculamos media, mediana y percentiles usando describe()
# Le pasamos una lista de percentiles para tener una visión más detallada
estadisticas = frecuencias_series.describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

print("\n--- Estadísticas Descriptivas y Percentiles ---")
print(estadisticas.round(2)) # Redondeamos a 2 decimales para que sea más legible

# 3. Calculamos la Moda (el número de apariciones que más se repite)
# (mode() devuelve una serie porque puede ser bimodal/multimodal, lo pasamos a lista)
moda = frecuencias_series.mode().tolist()
print("\n--- Medidas de Tendencia Central y Dispersión ---")
print(f"Moda: {moda}")

# 4. Otras métricas útiles: Varianza, Asimetría (Skewness) y Curtosis
varianza = frecuencias_series.var()
asimetria = frecuencias_series.skew()
curtosis = frecuencias_series.kurt()

print(f"Varianza: {varianza:.2f}")
print(f"Asimetría (Skewness): {asimetria:.2f} (Si es > 0, tiene cola hacia la derecha)")
print(f"Curtosis: {curtosis:.2f} (Pico de la distribución)")

# 5. Para dar contexto, mostramos los 10 códigos PANOSE reales más frecuentes
print("\n--- Top 10 Códigos PANOSE más presentes en el corpus ---")
for panose, count in panose_counter.most_common(10):
    print(f"PANOSE: {panose}  ->  {count} apariciones")

# VISUALIZACIÓN BÁSICA
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
# Histograma de las frecuencias, usando escala logarítmica si hay mucha diferencia
plt.hist(frecuencias, bins=50, color='skyblue', edgecolor='black', log=True)
plt.title('Distribución de frecuencias de los Códigos PANOSE (Escala Logarítmica)')
plt.xlabel('Número de apariciones en el dataset')
plt.ylabel('Cantidad de códigos PANOSE únicos')
plt.show()

In [ ]:
# Filtrado de name+panose por frecuencia

upper_limit = 19

filtered_panose_counter = {k: v for k, v in panose_counter.items() if v > 1 and v < upper_limit}  # Límite superior

filtered_panose_counter

In [ ]:
# Parejas de documentos con name+panose compartido
docs_parejas = []

for i, row1 in font_data_ole2.iterrows():
    for j, row2 in font_data_ole2.iterrows():
        if i < j:  # Evitar duplicados y auto-comparación
            if (row1['panose'], row1['name']) == (row2['panose'], row2['name']):
                
                # solo agregamos la pareja si el name+panose está en el filtered_panose_counter
                if (row1['panose'], row1['name']) in filtered_panose_counter:
                    docs_parejas.append((row1['sha256'], row2['sha256'], row1['panose'], row1['name']))

# Coincidencias de name+panose por pareja de documentos
docs_parejas_counter = Counter((doc1, doc2) for doc1, doc2, _, _ in docs_parejas)
docs_parejas_counter

In [ ]:
# a partir de docs_parejas_counter, creamos un grafo donde los nodos son los documentos y las aristas representan que comparten al menos un name+panose, 
# con el peso de la arista siendo el número de name+panose compartidos

G_ole2 = nx.Graph()
for (doc1, doc2), weight in docs_parejas_counter.items():
    G_ole2.add_edge(doc1, doc2, weight=weight)

# 4. Asignar atributos optimizado con diccionario
sha_to_apt = csv_full.set_index('sha256')['Normalized_Tag'].to_dict()

for node in G_ole2.nodes():
    # Formato
    ruta = f"/home/jupyter/binaries/{node}"
    G_ole2.nodes[node]['format'] = 'OLE2' if check_ole2(ruta) else 'Otro'
    
    # APT
    G_ole2.nodes[node]['apt'] = sha_to_apt.get(node, 'Desconocido')

# 5. Estadísticas
num_nodos = G_ole2.number_of_nodes()
num_aristas = G_ole2.number_of_edges()
print(f"El grafo tiene {num_nodos} nodos y {num_aristas} aristas.")

In [ ]:
# exportar para gephi
nx.write_gexf(G_ole2, "grafo_panose_ole2.gexf")

## Juntar Grafos

In [ ]:
# Contar las combinaciones de panose y name en todo el dataset
# conteo_grupo_ooxml = font_data_ooxml[['panose', 'name']].value_counts().reset_index(name='cantidad')
conteo_grupo_ooxml = font_data_ooxml[['csb', 'name']].value_counts().reset_index(name='cantidad')

# unir el panose y name con la cantidad de veces que aparecen juntos y meterlos en un counter
csb_counter_ooxml = Counter(zip(font_data_ooxml['csb'], font_data_ooxml['name']))

# ahora ponemos un diccionario donde la llave es el numero de apariciones y el valor es el numero de codigos panose con ese numero de apariciones
csb_distribution_ooxml = defaultdict(int)
for csb, count in csb_counter_ooxml.items():
    csb_distribution_ooxml[count] += 1

In [ ]:
upper_limit = 20

# filtered_panose_counter_ooxml = {k: v for k, v in panose_counter_ooxml.items() if v > 1 and v < upper_limit}  # Límite superior
filtered_csb_counter_ooxml = {k: v for k, v in csb_counter_ooxml.items() if v > 1 and v < upper_limit}  

# font_data_ooxml_filtered = font_data_ooxml[font_data_ooxml.apply(lambda row: (row['panose'], row['name']) in filtered_panose_counter_ooxml, axis=1)]
font_data_ooxml_filtered = font_data_ooxml[font_data_ooxml.apply(lambda row: (row['csb'], row['name']) in filtered_csb_counter_ooxml, axis=1)]

# añadimos una columna con OOXML que se llame 'format' y que tenga el valor 'OOXML'
font_data_ooxml_filtered['format'] = 'OOXML'

font_data_ooxml_filtered

In [ ]:
# Contar las combinaciones de panose y name en todo el dataset
# conteo_grupo_ole2 = font_data_ole2[['panose', 'name']].value_counts().reset_index(name='cantidad')
conteo_grupo_ole2 = font_data_ole2[['csb', 'name']].value_counts().reset_index(name='cantidad')

# unir el panose y name con la cantidad de veces que aparecen juntos y meterlos en un counter
# panose_counter_ole2 = Counter(zip(font_data_ole2['panose'], font_data_ole2['name']))
csb_counter_ole2 = Counter(zip(font_data_ole2['csb'], font_data_ole2['name']))

# ahora ponemos un diccionario donde la llave es el numero de apariciones y el valor es el numero de codigos panose con ese numero de apariciones
csb_distribution_ole2 = defaultdict(int)
for csb, count in csb_counter_ole2.items():
    csb_distribution_ole2[count] += 1

In [ ]:
upper_limit = 19

# filtered_panose_counter_ole2 = {k: v for k, v in panose_counter_ole2.items() if v > 1 and v < upper_limit}  # Límite superior
filtered_csb_counter_ole2 = {k: v for k, v in csb_counter_ole2.items() if v > 1 and v < upper_limit}  

# font_data_ole2_filtered = font_data_ole2[font_data_ole2.apply(lambda row: (row['panose'], row['name']) in filtered_panose_counter_ole2, axis=1)]
font_data_ole2_filtered = font_data_ole2[font_data_ole2.apply(lambda row: (row['csb'], row['name']) in filtered_csb_counter_ole2, axis=1)]

# dropeamos las columnas weight, is_truetype, family y pitch
font_data_ole2_filtered = font_data_ole2_filtered.drop(columns=['weight', 'is_truetype', 'family', 'pitch'])
# ahora añadimos una columna con OLE2 que se llame 'format' y que tenga el valor 'OLE2'
font_data_ole2_filtered['format'] = 'OLE2'

font_data_ole2_filtered

In [ ]:
# concatenamos los dos dataframes font_data_ooxml_filtered y font_data_ole2_filtered en uno solo
font_data_combined = pd.concat([font_data_ooxml_filtered, font_data_ole2_filtered], ignore_index=True)

font_data_combined

In [ ]:
# Parejas de documentos con name+panose compartido
docs_parejas = []

for i, row1 in font_data_combined.iterrows():
    for j, row2 in font_data_combined.iterrows():
        if i < j:  # Evitar duplicados y auto-comparación
            # if (row1['panose'], row1['name']) == (row2['panose'], row2['name']):
            if (row1['csb'], row1['name']) == (row2['csb'], row2['name']):
                
                # docs_parejas.append((row1['sha256'], row2['sha256'], row1['panose'], row1['name']))
                docs_parejas.append((row1['sha256'], row2['sha256'], row1['csb'], row1['name']))

# Coincidencias de name+panose por pareja de documentos
docs_parejas_counter = Counter((doc1, doc2) for doc1, doc2, _, _ in docs_parejas)
docs_parejas_counter

In [ ]:
# a partir de docs_parejas_counter, creamos un grafo donde los nodos son los documentos y las aristas representan que comparten al menos un name+panose, 
# con el peso de la arista siendo el número de name+panose compartidos

G = nx.Graph()
for (doc1, doc2), weight in docs_parejas_counter.items():
    G.add_edge(doc1, doc2, weight=weight)

# 4. Asignar atributos optimizado con diccionario
sha_to_apt = csv_full.set_index('sha256')['Normalized_Tag'].to_dict()

for node in G.nodes():
    # Formato lo miramos en font_data_combined en la columna 'format'
    format_value = font_data_combined.loc[font_data_combined['sha256'] == node, 'format'].values
    G.nodes[node]['format'] = format_value[0]
    
    # APT
    G.nodes[node]['apt'] = sha_to_apt.get(node, 'Desconocido')

# 5. Estadísticas
num_nodos = G.number_of_nodes()
num_aristas = G.number_of_edges()
print(f"El grafo tiene {num_nodos} nodos y {num_aristas} aristas.")

In [ ]:
# exportar para gephi
# nx.write_gexf(G, "grafo_panose_combined.gexf")
# nx.write_gexf(G, "grafo_usb_combined.gexf")
nx.write_gexf(G, "grafo_csb_combined.gexf")

# RSIDs Y CLSIDs

Extracción de identificadores internos de Word y construcción de grafos de coincidencias.

## RSIDs

### OOXML

In [ ]:
rsids_ooxml = []


font_data = []

for docname in docs:
    full_path = os.path.join(path, docname)

    print(f"Procesando archivo: {full_path}")

    if check_ooxml(full_path):
        print(f"Archivo {docname} es un documento OOXML válido.")
        print(f"Analizando PANOSE, USB y CSB en: {docname}")
        font_info = search_PANOSE_USB_CSB(full_path, verbose=True)
        if font_info:
            font_data.extend(font_info)





for n, docname in csv_full['sha256'].items():


    #if check_ooxml("/home/jupyter/binaries/" + docname):
    if check_ooxml("/home/jupyter/workspace/" + docname):
        #print(f"{docname}: ooxml")


        l = [] 

        try:
            #with zipfile.ZipFile("/home/jupyter/binaries/" + docname, 'r') as zip_ref:
            with zipfile.ZipFile("/home/jupyter/workspace/" + docname, 'r') as zip_ref:

                if 'word/settings.xml' in zip_ref.namelist():
                    contenido_document = zip_ref.read('word/settings.xml')
                    root = ET.fromstring(contenido_document.decode('utf-8', errors='ignore'))
                
                    ns_w = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'

                    for rsid_tag in root.findall(f'.//{ns_w}rsid'):
                        rsid_val = rsid_tag.get(f'{ns_w}val')
                        if rsid_val:
                            # LO GUARDAMOS EN MAYÚSCULAS PARA CRUZARLO CON OLE2
                            l.append(rsid_val.upper())

                # Imprimimos la lista directamente (eliminando repetidos con 'set')
                rsids_unicos = list(set(l))
                #print(f"{docname} - RSIDs encontrados: {rsids_unicos}")

                if rsids_unicos and len(rsids_unicos) > 0:
                    rsids_ooxml.append({
                        "sha256": docname,
                        "file_type": "ooxml",
                        "rsids": rsids_unicos
                    })
                
        except Exception as e:
            print(f"Error procesando el zip de {docname}: {e}")

### OLE2

In [ ]:
def extraer_rsids_ole2(doc_path):
    rsids_encontrados = set()
    
    try:
        with olefile.OleFileIO(doc_path) as ole:
            if not ole.exists('WordDocument'):
                return []
            wordstream = ole.openstream('WordDocument').read()
            
            if len(wordstream) < 1066:
                return []

            flag = struct.unpack("<H", wordstream[10:12])[0]
            table_name = "1Table" if (flag & 0x200) else "0Table"
            
            if not ole.exists(table_name):
                return []
            tablestream = ole.openstream(table_name).read()
            
            fcPlrsid  = struct.unpack("<I", wordstream[1058:1062])[0]
            lcbPlrsid = struct.unpack("<I", wordstream[1062:1066])[0]
            
            if fcPlrsid == 0 or lcbPlrsid == 0:
                return []

            # Tamaño del tablestream
            if fcPlrsid + lcbPlrsid > len(tablestream):
                return []

            plrsid_data = tablestream[fcPlrsid : fcPlrsid + lcbPlrsid]
            
            if len(plrsid_data) < 4:
                return []

            irsidMac = struct.unpack("<I", plrsid_data[0:4])[0]

            # irsidMac frente al tamaño real
            max_rsids = (len(plrsid_data) - 4) // 4
            if irsidMac > max_rsids:
                irsidMac = max_rsids  # documentos corruptos/manipulados

            offset = 4  # ✅ Solo 4 bytes de cabecera, no 24

            for _ in range(irsidMac):
                if offset + 4 > len(plrsid_data):
                    break
                rsid_val = struct.unpack("<I", plrsid_data[offset:offset+4])[0]
                rsids_encontrados.add(f"{rsid_val:08X}")
                offset += 4
                        
    except Exception as e:
        print(f"Error en OLE2 ({doc_path}): {e}")
        
    return list(rsids_encontrados)




rsids_ole2 = []

for n, docname in csv_full['sha256'].items():
    if check_ole2("/home/jupyter/binaries/" + docname):
        #print(f"{docname}: ole2")
        rsids = extraer_rsids_ole2("/home/jupyter/binaries/" + docname)
        #print(f"{docname} - RSIDs encontrados: {rsids}")

        if rsids and len(rsids) > 0:

            rsids_ole2.append({
                "sha256": docname,
                "file_type": "ole2",
                "rsids": rsids
            })

### Crear los DF
Los creamos por separado para poder hacer un analisis por separado y filtrarlos por separado y eso

In [ ]:
df_rsids_ooxml = pd.DataFrame(rsids_ooxml)
#df_rsids_ole2 = pd.DataFrame(rsids_ole2)

df_rsids_ooxml.head(5)

In [ ]:
rsids_ooxml

In [ ]:
# analizar la estructura de archivos de doc2.docx

with zipfile.ZipFile("/home/jupyter/workspace/doc2.docx", 'r') as zip_ref:
    print("Contenido del archivo doc2.docx:")
    for file_info in zip_ref.infolist():
        print(f"- {file_info.filename} (Tamaño: {file_info.file_size} bytes)")

In [ ]:
# analizar la estructura de archivos de doc1.docx

with zipfile.ZipFile("/home/jupyter/workspace/doc1.docx", 'r') as zip_ref:
    print("Contenido del archivo doc1.docx:")
    for file_info in zip_ref.infolist():
        print(f"- {file_info.filename} (Tamaño: {file_info.file_size} bytes)")

In [ ]:
# ver cuantos valores únicos hay de sha256 y compararlos con el numero de ooxml en csv_full
unique_sha256_count = df_rsids_ooxml['sha256'].nunique()

print(f"Valores únicos de sha256 en df_rsids_ooxml: {unique_sha256_count}")

total = 0
total_no_rsid_data = 0

for docname in csv_full['sha256'].items():
    if check_ooxml("/home/jupyter/binaries/" + docname[1]):
        total += 1
        if docname[1] not in df_rsids_ooxml['sha256'].values:
            total_no_rsid_data += 1

print(f"Total de documentos OOXML: {total}")
print(f"Total de documentos OOXML sin datos de fuente: {total_no_rsid_data}")



# ver cuantos valores únicos hay de sha256 y compararlos con el numero de ole2 en csv_full
unique_sha256_count = df_rsids_ole2['sha256'].nunique()

print(f"Valores únicos de sha256 en df_rsids_ole2: {unique_sha256_count}")

total = 0
total_no_rsid_data = 0

for docname in csv_full['sha256'].items():
    if check_ole2("/home/jupyter/binaries/" + docname[1]):
        total += 1
        if docname[1] not in df_rsids_ole2['sha256'].values:
            total_no_rsid_data += 1

print(f"Total de documentos OLE2: {total}")
print(f"Total de documentos OLE2 sin datos de fuente: {total_no_rsid_data}")

In [ ]:
# hacer un diccionario invertidos donde las llaves sean los rsids y los valores sean listas de sha256 que tienen ese rsid
dict_inversed_ooxml = defaultdict(list)
for _, row in df_rsids_ooxml.iterrows():
    for rsid in row['rsids']:
        dict_inversed_ooxml[rsid].append(row['sha256'])

dict_inversed_ole2 = defaultdict(list)
for _, row in df_rsids_ole2.iterrows():
    for rsid in row['rsids']:
        dict_inversed_ole2[rsid].append(row['sha256'])


# creamos un counter de los rsids, segun en el numero de docs que aparezcan
rsid_counter_ooxml = Counter({rsid: len(docs) for rsid, docs in dict_inversed_ooxml.items()})
rsid_counter_ole2 = Counter({rsid: len(docs) for rsid, docs in dict_inversed_ole2.items()})



In [ ]:
analyze_distribution(rsid_counter_ooxml)

In [ ]:
analyze_distribution(rsid_counter_ole2)

In [ ]:
min_freq_ooxml = 2
max_freq_ooxml = 14

min_freq_ole2 = 2
max_freq_ole2 = 25

# hacer una lista de rsids donde los rsids sean los que tengan una frecuencia de aparicion entre min_freq y max_freq
filtered_rsid_ooxml = [rsid for rsid, count in rsid_counter_ooxml.items() if min_freq_ooxml <= count <= max_freq_ooxml]
filtered_rsid_ole2 = [rsid for rsid, count in rsid_counter_ole2.items() if min_freq_ole2 <= count <= max_freq_ole2]

### Creacion del grafo
- 1- Grafo con pesos 1/N
- 2- Grafo con pesos 1

In [ ]:
# 1. Unificamos los RSIDs filtrados
todos_los_rsids_filtrados = set(filtered_rsid_ooxml) | set(filtered_rsid_ole2)

# 2. Unificamos los diccionarios inversos
dict_inversed_total = defaultdict(list)

for rsid, docs in dict_inversed_ooxml.items():
    dict_inversed_total[rsid].extend(docs)

for rsid, docs in dict_inversed_ole2.items():
    dict_inversed_total[rsid].extend(docs)

# 3. Construimos el grafo (Ahora es un solo bucle)
G_rsid = nx.Graph()

for rsid in todos_los_rsids_filtrados:
    docs = dict_inversed_total[rsid]
    
    # Requiere al menos 2 documentos
    if len(docs) > 1:
        weight = 1 / len(docs)
        for i in range(len(docs)):
            for j in range(i + 1, len(docs)):
                # docs[i] y docs[j] ahora pueden ser uno OLE2 y otro OOXML
                G_rsid.add_edge(docs[i], docs[j], weight=G_rsid.get_edge_data(docs[i], docs[j], {}).get('weight', 0) + weight)

# por cada nodo le agregamos el atributo 'format' que puede ser 'OOXML' o 'OLE2' segun el sha256 del nodo
# y un "apt" que es el valor de Normalized_Tag en csv_full
for node in G_rsid.nodes():
    # Formato lo miramos en csv_full en la columna 'sha256'
    format_value = 'OOXML' if check_ooxml(f"/home/jupyter/binaries/{node}") else 'OLE2'
    G_rsid.nodes[node]['format'] = format_value
    
    # APT
    G_rsid.nodes[node]['apt'] = sha_to_apt.get(node, 'Desconocido')


# Estadísticas
print(f"El grafo tiene {G_rsid.number_of_nodes()} nodos y {G_rsid.number_of_edges()} aristas.")


# exportar para gephi
nx.write_gexf(G_rsid, "grafo_rsid_combined_pesos1-N.gexf")

In [ ]:
# 1. Unificamos los RSIDs filtrados
todos_los_rsids_filtrados = set(filtered_rsid_ooxml) | set(filtered_rsid_ole2)

# 2. Unificamos los diccionarios inversos
dict_inversed_total = defaultdict(list)

for rsid, docs in dict_inversed_ooxml.items():
    dict_inversed_total[rsid].extend(docs)

for rsid, docs in dict_inversed_ole2.items():
    dict_inversed_total[rsid].extend(docs)

# 3. Construimos el grafo (Ahora es un solo bucle)
G_rsid = nx.Graph()

for rsid in todos_los_rsids_filtrados:
    docs = dict_inversed_total[rsid]
    
    # Requiere al menos 2 documentos
    if len(docs) > 1:
        weight = 1 #/ len(docs)
        for i in range(len(docs)):
            for j in range(i + 1, len(docs)):
                # docs[i] y docs[j] ahora pueden ser uno OLE2 y otro OOXML
                G_rsid.add_edge(docs[i], docs[j], weight=G_rsid.get_edge_data(docs[i], docs[j], {}).get('weight', 0) + weight)

# por cada nodo le agregamos el atributo 'format' que puede ser 'OOXML' o 'OLE2' segun el sha256 del nodo
# y un "apt" que es el valor de Normalized_Tag en csv_full
for node in G_rsid.nodes():
    # Formato lo miramos en csv_full en la columna 'sha256'
    format_value = 'OOXML' if check_ooxml(f"/home/jupyter/binaries/{node}") else 'OLE2'
    G_rsid.nodes[node]['format'] = format_value
    
    # APT
    G_rsid.nodes[node]['apt'] = sha_to_apt.get(node, 'Desconocido')


# Estadísticas
print(f"El grafo tiene {G_rsid.number_of_nodes()} nodos y {G_rsid.number_of_edges()} aristas.")


# exportar para gephi
nx.write_gexf(G_rsid, "grafo_rsid_combined.gexf")

## CLSIDs

### OOXML

In [ ]:
def extract_clsids_from_ooxml(ooxml_path):
    """
    Extrae CLSIDs de ficheros OOXML de forma exhaustiva y OPTIMIZADA.
    Busca en XMLs, rels y parsea streams OLE2 reales dentro del ZIP.
    """
    
    # CLSID canónico en texto (con guiones)
    regex_clsid_text = re.compile(
        r'\b([0-9A-Fa-f]{8}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{12})\b'
    )

    findings = []
    seen_global = set()  # deduplicación global por (clsid_upper, location)

    def add_text_findings(content: str, location: str, source_type: str):
        for m in regex_clsid_text.finditer(content):
            clsid = m.group(1).upper()
            if clsid == "00000000-0000-0000-0000-000000000000":
                continue
            key = (clsid, location)
            if key not in seen_global:
                seen_global.add(key)
                # Extraer contexto: 30 chars a cada lado
                start = max(0, m.start() - 30)
                end = min(len(content), m.end() + 30)
                ctx = content[start:end].replace('\n', ' ').replace('\r', '')
                findings.append({
                    "clsid": clsid,
                    "location": location,
                    "context": ctx.strip(),
                    "source_type": source_type
                })

    try:
        with zipfile.ZipFile(ooxml_path, 'r') as archive:
            for file_info in archive.infolist():
                fname = file_info.filename
                raw = archive.read(file_info)

                # ── 1. XMLs y RELs: búsqueda textual 
                if fname.endswith(('.xml', '.rels')):
                    text = raw.decode('utf-8', errors='ignore')
                    add_text_findings(text, fname, "xml_text")

                # ── 2. Binarios embebidos (OLE, EMF, WMF, docs anidados) ───────
                elif fname.endswith(('.bin', '.emf', '.wmf', '.ole')):
                    # Búsqueda textual por si hay CLSIDs en texto plano dentro del binario
                    text = raw.decode('utf-8', errors='ignore')
                    add_text_findings(text, fname, "bin_text")

                    # Extracción OLE real en memoria (Sustituye a la antigua búsqueda byte a byte)
                    if raw.startswith(b'\xd0\xcf\x11\xe0'): # Firma de OLE2
                        try:
                            with io.BytesIO(raw) as f:
                                with olefile.OleFileIO(f) as ole:
                                    for entry in ole.direntries:
                                        if entry and entry.clsid:
                                            clsid_str = str(entry.clsid).upper()
                                            if clsid_str != "00000000-0000-0000-0000-000000000000":
                                                key = (clsid_str, fname)
                                                if key not in seen_global:
                                                    seen_global.add(key)
                                                    findings.append({
                                                        "clsid": clsid_str,
                                                        "location": f"{fname} (Stream: {entry.name})",
                                                        "context": "OLE2_Metadata",
                                                        "source_type": "binary_ole_metadata"
                                                    })
                        except Exception:
                            pass # Ignoramos si el OLE está corrupto o no se puede parsear

                # ── 3. Cualquier otro fichero: búsqueda textual como fallback ──
                else:
                    try:
                        text = raw.decode('utf-8', errors='ignore')
                        add_text_findings(text, fname, "other_text")
                    except Exception:
                        pass

    except zipfile.BadZipFile:
        pass

    return findings

In [ ]:
dict_clsids_ooxml = {
    "sha256": [],
    "file_type": [],
    "clsids": []
}


print(f"Longitud de csv_full: {len(csv_full)}")
proc = 0



docs = ['doc1.docx', 'doc2.docx']




#for n, docname in csv_full['sha256'].items():

for docname in docs:


    proc += 1
    print(f"Procesando {proc}/{len(csv_full)}", end="\r")
    # if check_ooxml("/home/jupyter/binaries/" + docname):
    if check_ooxml("/home/jupyter/workspace/" + docname):
        #print(f"Procesando {docname} para extracción de CLSIDs...")
        try:
            #clsid_results = extract_clsids_from_ooxml("/home/jupyter/binaries/" + docname)
            clsid_results = extract_clsids_from_ooxml("/home/jupyter/workspace/" + docname)
            #print(f"Encontrados {len(clsid_results)} CLSIDs en {docname}:")
            for res in clsid_results:
                #print(f"  - CLSID: {res['clsid']} | Ubicación: {res['location']} | Fuente: {res['source_type']}")
                dict_clsids_ooxml["sha256"].append(docname)
                dict_clsids_ooxml["file_type"].append("OOXML")
                dict_clsids_ooxml["clsids"].append(res['clsid'])
        except Exception as e:
            print(f"Error procesando {docname}: {e}")

In [ ]:
dict_clsids_ooxml

### OLE2

In [ ]:
def find_clsids_binary(data: bytes, location: str) -> list:
    """
    Busca CLSIDs en formato binario little-endian (COM/OLE2 byte order).
    Los primeros 3 componentes van en LE, los últimos 2 en big-endian.
    """
    results = []
    seen = set()

    for i in range(len(data) - 15):
        chunk = data[i:i + 16]
        try:
            p1 = struct.unpack_from('<I', chunk, 0)[0]
            p2 = struct.unpack_from('<H', chunk, 4)[0]
            p3 = struct.unpack_from('<H', chunk, 6)[0]
            p4 = chunk[8:10].hex().upper()
            p5 = chunk[10:16].hex().upper()
            canonical = f"{p1:08X}-{p2:04X}-{p3:04X}-{p4}-{p5}"

            if canonical == "00000000-0000-0000-0000-000000000000":
                continue
            if p1 == 0 and p2 == 0 and p3 == 0:
                continue

            key = (canonical, location)
            if key not in seen:
                seen.add(key)
                results.append({
                    "clsid":       canonical,
                    "location":    location,
                    "context":     f"binary_offset_0x{i:X}",
                    "source_type": "binary_le"
                })
        except struct.error:
            continue

    return results


def extract_clsids_from_ole2(doc_path: str) -> list:
    """
    Extrae CLSIDs de ficheros OLE2 (DOC, XLS, PPT...) de forma exhaustiva.

    Fuentes cubiertas:
      - ole2_root_entry   : CLSID del directorio raíz (identifica la app creadora)
      - ole2_dir_entry    : CLSID de cada nodo del árbol CFB (storages con objetos)
      - ole2_compobj      : CLSID binario LE en el stream \\x01CompObj (offset 8-24)
      - ole2_embedded_obj : CLSID binario LE en \\x01CompObj y \\x01Ole de sub-storages
      - ole2_carving      : carving textual + triple decodificación sobre todos los streams

    Returns: list of dicts {clsid, location, context, source_type}
    """
    regex_clsid_text = re.compile(
        r'\b([0-9A-Fa-f]{8}-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{4}'
        r'-[0-9A-Fa-f]{4}-[0-9A-Fa-f]{12})\b'
    )
    NULL_CLSID = "00000000-0000-0000-0000-000000000000"

    findings    = []
    seen_global = set()

    def _add(clsid, location, context, source_type):
        clsid = clsid.upper()
        if clsid == NULL_CLSID:
            return
        key = (clsid, location)
        if key not in seen_global:
            seen_global.add(key)
            findings.append({
                "clsid":       clsid,
                "location":    location,
                "context":     context,
                "source_type": source_type
            })

    def _parse_clsid_le(chunk: bytes, location: str, source_type: str):
        """Parsea 16 bytes como CLSID little-endian y lo añade si es válido."""
        if len(chunk) < 16:
            return
        p1 = struct.unpack_from('<I', chunk, 0)[0]
        p2 = struct.unpack_from('<H', chunk, 4)[0]
        p3 = struct.unpack_from('<H', chunk, 6)[0]
        p4 = chunk[8:10].hex().upper()
        p5 = chunk[10:16].hex().upper()
        clsid = f"{p1:08X}-{p2:04X}-{p3:04X}-{p4}-{p5}"
        if p1 == 0 and p2 == 0 and p3 == 0:
            return
        _add(clsid, location, "compobj_offset_8", source_type)

    def _carving_text(content: str, location: str):
        """Carving textual con contexto de 30 chars a cada lado."""
        for m in regex_clsid_text.finditer(content):
            clsid = m.group(1).upper()
            start = max(0, m.start() - 30)
            end   = min(len(content), m.end() + 30)
            ctx   = content[start:end].replace('\n', ' ').replace('\r', '').strip()
            _add(clsid, location, ctx, "ole2_carving")

    if not olefile.isOleFile(doc_path):
        return [{"error": f"No es OLE2 válido: {doc_path}"}]

    try:
        with olefile.OleFileIO(doc_path) as ole:

            # FASE 1 — Fuentes estructurales del árbol CFB

            # 1a. CLSID del directorio raíz
            try:
                root_clsid = ole.root.clsid
                if root_clsid:
                    _add(root_clsid, "root_directory_entry",
                         "CFB root entry", "ole2_root_entry")
            except Exception:
                pass

            # 1b. CLSID de cada directory entry del árbol
            try:
                for entry in ole.direntries:
                    if entry is None:
                        continue
                    clsid = entry.clsid
                    if clsid:
                        _add(clsid, f"dir_entry:{entry.name}",
                             f"CFB dir entry type={entry.entry_type}",
                             "ole2_dir_entry")
            except Exception:
                pass

            # 1c. Stream \x01CompObj del root — CLSID en offset 8 (LE)
            try:
                for comp_path in ['\x01CompObj', ['\x01CompObj']]:
                    if ole.exists(comp_path):
                        comp_data = ole.openstream(comp_path).read()
                        if len(comp_data) >= 24:
                            _parse_clsid_le(
                                comp_data[8:24],
                                "CompObj_stream",
                                "ole2_compobj"
                            )
                        break
            except Exception:
                pass

            # 1d. \x01CompObj y \x01Ole dentro de sub-storages (objetos embebidos)
            try:
                for entry in ole.direntries:
                    if entry is None or entry.entry_type != 1:  # 1 = STGTY_STORAGE
                        continue
                    storage_name = entry.name
                    for sub_stream in ['\x01CompObj', '\x01Ole']:
                        sub_path = [storage_name, sub_stream]
                        try:
                            if ole.exists(sub_path):
                                sub_data = ole.openstream(sub_path).read()
                                # Búsqueda binaria LE completa sobre el stream
                                for hit in find_clsids_binary(
                                    sub_data,
                                    f"{storage_name}/{sub_stream}"
                                ):
                                    _add(hit["clsid"], hit["location"],
                                         hit["context"], "ole2_embedded_obj")
                        except Exception:
                            continue
            except Exception:
                pass

            # FASE 2 — Carving textual sobre todos los streams
            # Triple decodificación para cubrir desincronización UTF-16LE
            try:
                for stream_path in ole.listdir():
                    try:
                        raw_data = ole.openstream(stream_path).read()
                        if not raw_data:
                            continue
                        stream_name = "/".join(stream_path)
                        combined = (
                            raw_data.decode('utf-8', errors='ignore')    + " \n " +
                            raw_data.decode('utf-16le', errors='ignore') + " \n " +
                            raw_data[1:].decode('utf-16le', errors='ignore')
                        )
                        _carving_text(combined, stream_name)
                    except Exception:
                        continue
            except Exception:
                pass

    except Exception as e:
        return [{"error": f"Error fatal abriendo OLE2: {str(e)}"}]

    return findings

In [ ]:
dict_clsids_ole2 = {
    "sha256": [],
    "file_type": [],
    "clsids": []
}


proc = 0

for n, docname in csv_full['sha256'].items():
    proc += 1
    print(f"Procesando {proc}/{len(csv_full)}", end="\r")
    if check_ole2("/home/jupyter/binaries/" + docname):
        try:
            clsid_results = extract_clsids_from_ole2("/home/jupyter/binaries/" + docname)
            for res in clsid_results:
                if len(res['clsid']) > 0:
                    dict_clsids_ole2["sha256"].append(docname)
                    dict_clsids_ole2["file_type"].append("OLE2")
                    dict_clsids_ole2["clsids"].append(res['clsid'])
        except Exception as e:
            print(f"Error procesando {docname}: {e}")

### Crear y Procesar los df

In [ ]:
df_clsids_ooxml = pd.DataFrame(dict_clsids_ooxml)
df_clsids_ole2  = pd.DataFrame(dict_clsids_ole2)

df_clsids_ooxml

In [ ]:
# ESTADÍSTICAS PARA OOXML

# 1. Convertimos la columna a un set (búsqueda ultra rápida)
set_sha256_ooxml = set(df_clsids_ooxml['sha256'])
unique_sha256_count_ooxml = len(set_sha256_ooxml)

print(f"Valores únicos de sha256 en df_clsids_ooxml: {unique_sha256_count_ooxml}")

total_ooxml = 0
total_no_clsid_data_ooxml = 0

for _, doc_hash in csv_full['sha256'].items():
    if check_ooxml(f"/home/jupyter/binaries/{doc_hash}"):
        total_ooxml += 1
        # 2. Buscamos en el set en lugar de en .values
        if doc_hash not in set_sha256_ooxml:
            total_no_clsid_data_ooxml += 1

print(f"Total de documentos OOXML: {total_ooxml}")
print(f"Total de documentos OOXML sin datos de CLSID: {total_no_clsid_data_ooxml}\n")


# ESTADÍSTICAS PARA OLE2

# 1. Convertimos la columna a un set
set_sha256_ole2 = set(df_clsids_ole2['sha256'])
unique_sha256_count_ole2 = len(set_sha256_ole2)

print(f"Valores únicos de sha256 en df_clsids_ole2: {unique_sha256_count_ole2}")

total_ole2 = 0
total_no_clsid_data_ole2 = 0

for _, doc_hash in csv_full['sha256'].items():
    if check_ole2(f"/home/jupyter/binaries/{doc_hash}"):
        total_ole2 += 1
        # 2. Buscamos en el set
        if doc_hash not in set_sha256_ole2:
            total_no_clsid_data_ole2 += 1

print(f"Total de documentos OLE2: {total_ole2}")
print(f"Total de documentos OLE2 sin datos de CLSID: {total_no_clsid_data_ole2}")

In [ ]:
from collections import defaultdict, Counter

# 1. Documentos únicos mediante set
dict_inversed_clsid_ooxml = defaultdict(set)
dict_inversed_clsid_ole2 = defaultdict(set)

# Nombre de la columna
col_clsid_ooxml = 'clsids' if 'clsids' in df_clsids_ooxml.columns else 'clsid'
col_clsid_ole2 = 'clsids' if 'clsids' in df_clsids_ole2.columns else 'clsid'

# 2. Iteramos usando zip() (Ultra rápido)
for clsid, doc_hash in zip(df_clsids_ooxml[col_clsid_ooxml], df_clsids_ooxml['sha256']):
    dict_inversed_clsid_ooxml[clsid].add(doc_hash)

for clsid, doc_hash in zip(df_clsids_ole2[col_clsid_ole2], df_clsids_ole2['sha256']):
    dict_inversed_clsid_ole2[clsid].add(doc_hash)

# 3. Creamos los Counters (renombrados a clsid_counter)
# Nota: Ahora dict_inversed_clsid_ooxml tiene sets, pero para el grafo 
# Puede usarse una lista o un conjunto
# o convertirlo a lista al vuelo: {clsid: list(docs) for ...}
clsid_counter_ooxml = Counter({clsid: len(docs) for clsid, docs in dict_inversed_clsid_ooxml.items()})
clsid_counter_ole2 = Counter({clsid: len(docs) for clsid, docs in dict_inversed_clsid_ole2.items()})

# Cinco CLSIDs más comunes
print("Top 5 CLSIDs en OOXML:")
for clsid, count in clsid_counter_ooxml.most_common(5):
    print(f"  {clsid}: {count} documentos")

print("\nTop 5 CLSIDs en OLE2:")
for clsid, count in clsid_counter_ole2.most_common(5):
    print(f"  {clsid}: {count} documentos")

In [ ]:
analyze_distribution(clsid_counter_ooxml)

In [ ]:
analyze_distribution(clsid_counter_ole2)

In [ ]:
min_freq_ooxml = 2
max_freq_ooxml = 17

min_freq_ole2 = 2
max_freq_ole2 = 9

# hacer una lista de rsids donde los rsids sean los que tengan una frecuencia de aparicion entre min_freq y max_freq
filtered_clsids_ooxml = [clsid for clsid, count in clsid_counter_ooxml.items() if min_freq_ooxml <= count <= max_freq_ooxml]
filtered_clsids_ole2 = [clsid for clsid, count in clsid_counter_ole2.items() if min_freq_ole2 <= count <= max_freq_ole2]

In [ ]:
# 1. Unificamos TODOS los diccionarios inversos primero (sin filtrar)
dict_inversed_total = defaultdict(set)

for clsid, docs in dict_inversed_clsid_ooxml.items():
    dict_inversed_total[clsid].update(docs)

for clsid, docs in dict_inversed_clsid_ole2.items():
    dict_inversed_total[clsid].update(docs)

# 2. Ahora aplicamos el filtro de frecuencia GLOBAL sobre el diccionario unificado
# Queremos CLSIDs que aparezcan en más de 1 documento (para conectar)
# pero en menos de, por ejemplo, 40 documentos en TOTAL de todo el corpus.
min_freq_global = 2
max_freq_global = 40  # Límite superior

clsids_finales_filtrados = [
    clsid for clsid, docs in dict_inversed_total.items()
    if min_freq_global <= len(docs) <= max_freq_global
]

### Creacion del Grafo

In [ ]:
# 1. Unificamos los CLSIDs filtrados
todos_los_clsids_filtrados = set(filtered_clsids_ooxml) | set(filtered_clsids_ole2)

# 2. Unificamos los diccionarios inversos
dict_inversed_total = defaultdict(list)

for clsid, docs in dict_inversed_clsid_ooxml.items():
    dict_inversed_total[clsid].extend(docs)

for clsid, docs in dict_inversed_clsid_ole2.items():
    dict_inversed_total[clsid].extend(docs)

# 3. Construimos el grafo (Ahora es un solo bucle)
G_clsid = nx.Graph()

for clsid in todos_los_clsids_filtrados:
    docs = dict_inversed_total[clsid]
    
    # Requiere al menos 2 documentos
    if len(docs) > 1:
        weight = 1 / len(docs)
        for i in range(len(docs)):
            for j in range(i + 1, len(docs)):
                # docs[i] y docs[j] ahora pueden ser uno OLE2 y otro OOXML
                G_clsid.add_edge(docs[i], docs[j], weight=G_clsid.get_edge_data(docs[i], docs[j], {}).get('weight', 0) + weight)


sha_to_apt = csv_full.set_index('sha256')['Normalized_Tag'].to_dict()


# por cada nodo le agregamos el atributo 'format' que puede ser 'OOXML' o 'OLE2' segun el sha256 del nodo
# y un "apt" que es el valor de Normalized_Tag en csv_full
for node in G_clsid.nodes():
    # Formato lo miramos en csv_full en la columna 'sha256'
    format_value = 'OOXML' if check_ooxml(f"/home/jupyter/binaries/{node}") else 'OLE2'
    G_clsid.nodes[node]['format'] = format_value
    
    # APT
    G_clsid.nodes[node]['apt'] = sha_to_apt.get(node, 'Desconocido')


# Estadísticas
print(f"El grafo tiene {G_clsid.number_of_nodes()} nodos y {G_clsid.number_of_edges()} aristas.")


# exportar para gephi
nx.write_gexf(G_clsid, "grafo_clsid_combined.gexf")

Grafo hecho con pesos 1/N

In [ ]:
# 3. Construimos el grafo
G_clsid = nx.Graph()

for clsid in clsids_finales_filtrados:
    docs = list(dict_inversed_total[clsid])
    weight = 1.0 / len(docs)
    
    # Generamos las parejas
    for doc1, doc2 in combinations(docs, 2):
        current_weight = G_clsid.get_edge_data(doc1, doc2, default={}).get('weight', 0.0)
        G_clsid.add_edge(doc1, doc2, weight=current_weight + weight)

# 4. Asignamos Atributos (Sin ir al disco duro)
set_sha256_ooxml = set(df_clsids_ooxml['sha256'])
sha_to_apt = csv_full.set_index('sha256')['Normalized_Tag'].to_dict()

for node in list(G_clsid.nodes()):
    G_clsid.nodes[node]['format'] = 'OOXML' if node in set_sha256_ooxml else 'OLE2'
    G_clsid.nodes[node]['apt'] = sha_to_apt.get(node, 'Desconocido')

print(f"El grafo tiene {G_clsid.number_of_nodes()} nodos y {G_clsid.number_of_edges()} aristas.")

In [ ]:
# exportar para gephi
nx.write_gexf(G_clsid, "grafo_clsid_combined.gexf")

# Imagenes

Extracción de imágenes, comparación perceptual y señales de atribución.

## Extraccion OOXML y OLE2

In [ ]:



OOXML_IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', 
                           '.tiff', '.tif', '.wmf', '.emf', '.svg'}

def extract_images_ooxml(path):
    """
    Extrae todas las imágenes de un archivo OOXML.
    Las imágenes están en word/media/, xl/media/, ppt/media/
    """
    images = []
    doc_sha256 = os.path.basename(path)

    try:
        with zipfile.ZipFile(path, 'r') as z:
            media_files = [
                f for f in z.namelist()
                if '/media/' in f and Path(f).suffix.lower() in OOXML_IMAGE_EXTENSIONS
            ]

            for media_path in media_files:
                img_bytes = z.read(media_path)
                images.append({
                    'doc_sha256':  doc_sha256,
                    'format':      'ooxml',
                    'source_path': media_path,
                    'filename':    os.path.basename(media_path),
                    'extension':   Path(media_path).suffix.lower(),
                    'size_bytes':  len(img_bytes),
                    'md5':         hashlib.md5(img_bytes).hexdigest(),
                    'sha256':      hashlib.sha256(img_bytes).hexdigest(),
                    'data':        img_bytes,   # raw bytes para análisis posterior
                    'phash':       imagehash.phash(Image.open(BytesIO(img_bytes))),  # perceptual hash para comparación visual
                    'dhash':       imagehash.dhash(Image.open(BytesIO(img_bytes))),  # difference hash para comparación visual
                })

    except Exception as e:
        # print(f"Error OOXML {path}: {e}")
        pass

    return images


# ─────────────────────────────────────────────
# OLE2
# ─────────────────────────────────────────────

# Magic bytes → extensión
OLE2_IMAGE_MAGIC = {
    b'\x89PNG':         '.png',
    b'\xff\xd8\xff':    '.jpg',
    b'GIF8':            '.gif',
    b'BM':              '.bmp',
    b'\x01\x00\x00\x00': '.emf',   # EMF
    b'\xd7\xcd\xc6\x9a': '.wmf',   # WMF (placeable)
    b'\x00\x00\x01\x00': '.ico',
}

# Streams OLE2 que suelen contener imágenes
OLE2_IMAGE_STREAMS = [
    'WordDocument',      # .doc — imágenes embebidas en el stream principal
    'Pictures',          # stream dedicado en algunos .ppt
    '\x05SummaryInformation',  # a veces tiene thumbnail
]

def detect_image_format(data):
    """Detecta el formato de imagen por magic bytes"""
    for magic, ext in OLE2_IMAGE_MAGIC.items():
        if data[:len(magic)] == magic:
            return ext
    return None




def is_valid_emf(data):
    """
    Valida que un chunk sea realmente un EMF.
    El header EMF (EMR_HEADER record) tiene estructura fija:
      - Bytes 0-3:  record type = 0x00000001 (EMR_HEADER)
      - Bytes 4-7:  record size >= 88 bytes
      - Bytes 8-15: bounding box (4 signed ints, pueden ser cualquier valor)
      - Bytes 40-43: signature = 0x464D4520 (' EMF') ← clave
      - Bytes 44-47: version
    """
    if len(data) < 88:
        return False
    
    import struct
    
    # record type debe ser exactamente 1
    rec_type = struct.unpack_from('<I', data, 0)[0]
    if rec_type != 1:
        return False
    
    # record size debe ser >= 88 y razonable
    rec_size = struct.unpack_from('<I', data, 4)[0]
    if rec_size < 88 or rec_size > len(data):
        return False
    
    # signature ' EMF' en offset 40
    signature = struct.unpack_from('<I', data, 40)[0]
    if signature != 0x464D4520:
        return False
    
    return True


def is_valid_wmf(data):
    """
    WMF placeable: magic = 0x9AC2D7CD + header de 18 bytes fijo
    WMF estándar: magic word = 1 o 2 (memory/disk metafile)
    """
    if len(data) < 18:
        return False
    
    import struct
    
    # WMF placeable
    magic = struct.unpack_from('<I', data, 0)[0]
    if magic == 0x9AC2D7CD:
        return True
    
    # WMF estándar: word 0 = 1 (memory) o 2 (disk)
    wmf_type = struct.unpack_from('<H', data, 0)[0]
    wmf_version = struct.unpack_from('<H', data, 2)[0]
    if wmf_type in (1, 2) and wmf_version in (0x0100, 0x0300):
        return True
    
    return False




def extract_images_from_bytes(data, min_size=512):
    """
    Busca imágenes embebidas dentro de un stream binario
    escaneando por magic bytes. Útil para WordDocument y Pictures.
    """
    found = []
    
    # Patrones de inicio de imagen
    patterns = [
        (b'\x89PNG\r\n\x1a\n', '.png'),
        (b'\xff\xd8\xff',       '.jpg'),
        (b'GIF87a',             '.gif'),
        (b'GIF89a',             '.gif'),
        (b'\x01\x00\x00\x00',  '.emf'),
        (b'\xd7\xcd\xc6\x9a',  '.wmf'),
    ]
    
    for magic, ext in patterns:
        offset = 0
        while True:
            pos = data.find(magic, offset)
            if pos == -1:
                break
            
            # Extraer chunk desde el magic hasta el siguiente magic o fin
            chunk = data[pos:]
            
            # Para PNG: buscar IEND
            if ext == '.png':
                end = chunk.find(b'IEND\xaeB`\x82')
                if end != -1:
                    chunk = chunk[:end + 8]
                    
            # Para JPEG: buscar EOI marker
            elif ext == '.jpg':
                end = chunk.find(b'\xff\xd9')
                if end != -1:
                    chunk = chunk[:end + 2]
            

            if ext == '.emf' and not is_valid_emf(chunk):
                offset = pos + len(magic)
                continue

            if ext == '.wmf' and not is_valid_wmf(chunk):
                offset = pos + len(magic)
                continue


            if len(chunk) >= min_size:
                found.append((chunk, ext, pos))
            
            offset = pos + len(magic)
    
    return found


def extract_images_ole2(path):
    """
    Extrae todas las imágenes de un archivo OLE2.
    Estrategia: 
      1. Stream 'Pictures' (PowerPoint)
      2. Stream '1Table'/'0Table' + WordDocument (Word)
      3. Scan de todos los streams por magic bytes
    """
    images = []
    doc_sha256 = os.path.basename(path)

    try:
        ole = olefile.OleFileIO(path)
        all_streams = ole.listdir(streams=True, storages=False)

        for stream_path in all_streams:
            stream_name = stream_path[-1]
            
            try:
                data = ole.openstream(stream_path).read()
            except Exception:
                continue

            # ── Estrategia 1: stream 'Pictures' (PowerPoint) ──────────
            if stream_name == 'Pictures':
                # El stream Pictures de PPT tiene un header por imagen
                # Formato: [record_type(2)] [record_version(2)] [record_len(4)] [data]
                offset = 0
                img_idx = 0
                while offset < len(data) - 8:
                    try:
                        rec_type = int.from_bytes(data[offset:offset+2], 'little')
                        rec_len  = int.from_bytes(data[offset+4:offset+8], 'little')
                        
                        # Tipos de imagen en PPT: 0xF018-0xF01F
                        if 0xF018 <= rec_type <= 0xF01F and rec_len > 0:
                            img_data = data[offset+8:offset+8+rec_len]
                            
                            # Saltar header BLIP (16 o 17 bytes según tipo)
                            blip_offset = 17 if rec_type in (0xF01A, 0xF01B, 0xF01D, 0xF01F) else 25
                            img_data = img_data[blip_offset:] if len(img_data) > blip_offset else img_data
                            
                            ext = detect_image_format(img_data) or '.bin'
                            if len(img_data) > 512:
                                images.append({
                                    'doc_sha256':  doc_sha256,
                                    'format':      'ole2',
                                    'source_path': f"Pictures[{img_idx}]",
                                    'filename':    f"picture_{img_idx}{ext}",
                                    'extension':   ext,
                                    'size_bytes':  len(img_data),
                                    'md5':         hashlib.md5(img_data).hexdigest(),
                                    'sha256':      hashlib.sha256(img_data).hexdigest(),
                                    'data':        img_data,
                                    'phash':       imagehash.phash(Image.open(BytesIO(img_data))),
                                    'dhash':       imagehash.dhash(Image.open(BytesIO(img_data)))
                                })
                                img_idx += 1
                        
                        offset += 8 + rec_len
                    except Exception:
                        break

            # ── Estrategia 2: scan por magic bytes en cualquier stream ─
            else:
                found = extract_images_from_bytes(data)
                for idx, (img_data, ext, pos) in enumerate(found):
                    images.append({
                        'doc_sha256':  doc_sha256,
                        'format':      'ole2',
                        'source_path': f"{'/'.join(stream_path)}@{pos}",
                        'filename':    f"{'_'.join(stream_path)}_{idx}{ext}",
                        'extension':   ext,
                        'size_bytes':  len(img_data),
                        'md5':         hashlib.md5(img_data).hexdigest(),
                        'sha256':      hashlib.sha256(img_data).hexdigest(),
                        'data':        img_data,
                        'phash':       imagehash.phash(Image.open(BytesIO(img_data))),
                        'dhash':       imagehash.dhash(Image.open(BytesIO(img_data)))
                    })

        ole.close()

    except Exception as e:
        # print(f"Error OLE2 {path}: {e}")
        pass

    # Deduplicar por MD5 (el scan puede encontrar la misma imagen en varios streams)
    seen = set()
    unique = []
    for img in images:
        if img['md5'] not in seen:
            seen.add(img['md5'])
            unique.append(img)

    return unique


# ─────────────────────────────────────────────
# Dispatcher unificado
# ─────────────────────────────────────────────

def extract_images(path):
    try:
        with open(path, 'rb') as f:
            magic = f.read(4)
    except Exception:
        return []

    if magic == b'PK\x03\x04':
        return extract_images_ooxml(path)
    elif magic[:4] == b'\xd0\xcf\x11\xe0':
        return extract_images_ole2(path)
    return []




In [ ]:
all_images = []

# longitud de csv_full['sha256']
print(f"Total documentos en csv_full: {len(csv_full['sha256'])}")

total = 0


for docname in csv_full['sha256']:
    total += 1

    # printear en la misma línea el progreso cada 10 documentos
    if total % 10 == 0:
        print(f"Procesados {total}/{len(csv_full['sha256'])} documentos", end='\r')

    path = f"/home/jupyter/binaries/{docname}"
    if os.path.exists(path):
        imgs = extract_images(path)
        if imgs != []:
            all_images.extend(imgs)


# DataFrame sin los bytes (para análisis)
df_images = pd.DataFrame([
    {k: v for k, v in img.items() if k != 'data'}
    for img in all_images
])

In [ ]:
def get_doc_format(sha256):
    path = f"/home/jupyter/binaries/{sha256}"
    try:
        with open(path, 'rb') as f:
            magic = f.read(4)
    except Exception:
        return 'unknown'
    
    if magic == b'PK\x03\x04':
        return 'ooxml'
    elif magic[:4] == b'\xd0\xcf\x11\xe0':
        return 'ole2'
    return 'unknown'


# Añadir columna format al df_images
df_images['doc_format'] = df_images['doc_sha256'].map(get_doc_format)


df_images

In [ ]:
# ver cuantos doc_sha256 únicos hay en df_images de cada format
# compararlo con el numero de documentos OOXML y OLE2 que hay en csv_full
unique_docs_in_images = df_images.groupby('doc_format')['doc_sha256'].nunique()
print("Documentos únicos con imágenes extraídas por formato:")
print(unique_docs_in_images)

# Contar documentos en csv_full por formato (no tiene columna de formato, así que hay que iterar usando check_ooxml y check_ole2)
total_ooxml = sum(1 for docname in csv_full['sha256'] if check_ooxml(f"/home/jupyter/binaries/{docname}"))
total_ole2 = sum(1 for docname in csv_full['sha256'] if check_ole2(f"/home/jupyter/binaries/{docname}"))

print(f"Total documentos OOXML en csv_full: {total_ooxml}")
print(f"Total documentos OLE2 en csv_full: {total_ole2}")

In [ ]:
def hamming_pairs(df, hash_col, threshold=5):
    """
    Compara todos los pares de imágenes por distancia de Hamming.
    Devuelve un DataFrame con los pares cuya distancia <= threshold.
    """
    rows = df[['sha256', 'doc_sha256', 'doc_format', 'filename', hash_col]].dropna().to_dict('records')
    
    results = []
    total = len(rows)
    
    for i, (a, b) in enumerate(combinations(range(total), 2)):
        img_a = rows[a]
        img_b = rows[b]

        # Saltar si las dos imágenes son del mismo documento
        if img_a['doc_sha256'] == img_b['doc_sha256']:
            continue
        
        dist = img_a[hash_col] - img_b[hash_col]  # imagehash soporta resta directa → Hamming
        
        if dist <= threshold:
            results.append({
                'sha256_a':     img_a['sha256'],
                'doc_sha256_a': img_a['doc_sha256'],
                'filename_a':   img_a['filename'],
                'sha256_b':     img_b['sha256'],
                'doc_sha256_b': img_b['doc_sha256'],
                'filename_b':   img_b['filename'],
                'doc_format_a': img_a['doc_format'],
                'doc_format_b': img_b['doc_format'],
                hash_col:       dist,
                'cross_format':    img_a['doc_format'] != img_b['doc_format'],
            })
    
    return pd.DataFrame(results).sort_values(hash_col) if results else pd.DataFrame()


# ── Ejecutar para phash ────────────────────────────────────────────────────────
print("Comparando phash...")
df_phash_pairs = hamming_pairs(df_images, 'phash', threshold=5)
print(f"  Pares similares (phash ≤ 5): {len(df_phash_pairs)}")

# ── Ejecutar para dhash ────────────────────────────────────────────────────────
print("Comparando dhash...")
df_dhash_pairs = hamming_pairs(df_images, 'dhash', threshold=5)
print(f"  Pares similares (dhash ≤ 5): {len(df_dhash_pairs)}")

## Grafo y Filtrado
Se puede cambiar el phash y el dhash

In [ ]:
DHASH_THRESHOLD = 4
PHASH_THRESHOLD = 4

# Filtrar pares dhash cross-format con distancia < 4
df_graph = df_phash_pairs[
    #(df_dhash_pairs['cross_format']) &  <- solo si queremos relaciones entre formatos distintos
    #(df_dhash_pairs['dhash'] < DHASH_THRESHOLD)
    (df_phash_pairs['phash'] < PHASH_THRESHOLD)
].copy()

# Número de imágenes por doc (para normalizar)
imgs_per_doc = df_images.groupby('doc_sha256').size().to_dict()

# Construir grafo
G = nx.Graph()

# Nodos: un nodo por doc que aparezca en algún par
all_docs = set(df_graph['doc_sha256_a']) | set(df_graph['doc_sha256_b'])
for doc in all_docs:
    G.add_node(doc,
        format=df_images.loc[df_images['doc_sha256'] == doc, 'doc_format'].iloc[0],
        n_images=imgs_per_doc.get(doc, 0)
    )

# Aristas: agrupar pares por (doc_a, doc_b) y calcular peso normalizado
edge_counts = (
    df_graph
    .groupby(['doc_sha256_a', 'doc_sha256_b'])
    .size()
    .reset_index(name='n_pairs')
)

for _, row in edge_counts.iterrows():
    doc_a = row['doc_sha256_a']
    doc_b = row['doc_sha256_b']
    n_a   = imgs_per_doc.get(doc_a, 1)
    n_b   = imgs_per_doc.get(doc_b, 1)
    weight = row['n_pairs'] / (n_a * n_b) ** 0.5

    G.add_edge(doc_a, doc_b,
        weight=round(weight, 6),
        n_pairs=int(row['n_pairs'])
    )


# Mapear sha256 → Normalized_Tag
apt_map = csv_full.set_index('sha256')['Normalized_Tag'].to_dict()

# Añadir atributo apt a cada nodo
for doc in G.nodes():
    G.nodes[doc]['apt'] = apt_map.get(doc, 'unknown')



print(f"Nodos: {G.number_of_nodes()}  |  Aristas: {G.number_of_edges()}")

# Exportar a GEXF (Gephi lo abre nativamente, preserva atributos de nodos y aristas)
# nx.write_gexf(G, "graph_dhash.gexf")
# print("Exportado: graph_dhash.gexf")

nx.write_gexf(G, "graph_phash.gexf")
print("Exportado: graph_phash.gexf")

In [ ]:
# crear un dataset con el sha256 de los docs, de las imgs, el formato del doc y la data de la imagen
df_imagedata = df_images[['doc_sha256', 'sha256', 'doc_format']].drop_duplicates()

# añadir la columna 'data' con los bytes de la imagen
# data esta en el diccionario all_images, que tiene un sha256 de la imagen y los bytes
sha_to_data = {img['sha256']: img['data'] for img in all_images}
df_imagedata['data'] = df_imagedata['sha256'].map(sha_to_data)

df_imagedata

In [ ]:

def process_image_bytes_from_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Procesa imágenes cuyos bytes ya están en el DataFrame.
    
    Espera columnas: doc_sha256, sha256, doc_format, data (bytes)
    """
    results = []

    for _, row in df.iterrows():
        record = {
            'doc_sha256': row['doc_sha256'],
            'img_sha256': row['sha256'],
            'doc_format': row['doc_format'],
        }

        raw: bytes = row['data']

        # Si viene como string tipo b'\xff\xd8...' hay que parsearlo
        if isinstance(raw, str):
            raw = _parse_bytes_literal(raw)

        if not raw:
            record['error'] = 'bytes vacíos'
            results.append(record)
            continue

        # ── Detectar formato real por magic bytes ──────────────────
        record['magic_format'] = _detect_format(raw)

        # ── Abrir con Pillow para metadatos ────────────────────────
        try:
            img = Image.open(io.BytesIO(raw))
            record['width_px']   = img.width
            record['height_px']  = img.height
            record['mode']       = img.mode        # RGB, RGBA, P…
            record['img_format'] = img.format      # PNG, JPEG, BMP…

            # DPI si está disponible (no siempre presente)
            dpi = img.info.get('dpi')
            if dpi:
                record['dpi_x'] = dpi[0]
                record['dpi_y'] = dpi[1]
                record['width_cm']  = round(img.width  / dpi[0] * 2.54, 2)
                record['height_cm'] = round(img.height / dpi[1] * 2.54, 2)

            # Metadatos EXIF si existen
            exif = img._getexif() if hasattr(img, '_getexif') else None
            if exif:
                record['exif_keys'] = list(exif.keys())

        except Exception as e:
            record['error'] = str(e)

        results.append(record)

    return pd.DataFrame(results)


# ── Helpers ───────────────────────────────────────────────────────────────────

MAGIC_BYTES = {
    b'\xff\xd8\xff':        'JPEG',
    b'\x89PNG':             'PNG',
    b'GIF8':                'GIF',
    b'BM':                  'BMP',
    b'II*\x00':             'TIFF (LE)',
    b'MM\x00*':             'TIFF (BE)',
    b'RIFF':                'WEBP/AVI',
    b'\x00\x00\x01\x00':   'ICO',
    b'%PDF':                'PDF',
}

def _detect_format(data: bytes) -> str:
    for magic, fmt in MAGIC_BYTES.items():
        if data[:len(magic)] == magic:
            return fmt
    return 'unknown'


def _parse_bytes_literal(s: str) -> bytes:
    """
    Convierte el string  b'\\xff\\xd8\\xff...'  a bytes reales.
    Necesario si pandas leyó los bytes como repr en lugar de objeto bytes.
    """
    # Quitar el prefijo b' y el sufijo '
    s = s.strip()
    if s.startswith("b'") or s.startswith('b"'):
        s = s[2:-1]
    return s.encode('raw_unicode_escape').decode('unicode_escape').encode('latin-1')

In [ ]:
df_ooxml = df_imagedata[df_imagedata['doc_format'] == 'ooxml']
df_ole2  = df_imagedata[df_imagedata['doc_format'] == 'ole2']   # necesitaría tratamiento aparte

resultado = process_image_bytes_from_df(df_ooxml)

In [ ]:
for row in resultado.itertuples(index=False):
    print(f"Doc: {row.doc_sha256}, Img: {row.img_sha256}, Format: {row.img_format}, Size: {row.width_px}x{row.height_px}, DPI: ({getattr(row, 'dpi_x', 'N/A')}, {getattr(row, 'dpi_y', 'N/A')})")

In [ ]:
import zipfile, os, re, io, hashlib
import xml.etree.ElementTree as ET
import pandas as pd
from PIL import Image
from PIL.ExifTags import TAGS
import struct
from dataclasses import dataclass, field
from typing import Optional

# ── Namespaces ────────────────────────────────────────────────────────────────
NS = {
    'a':   'http://schemas.openxmlformats.org/drawingml/2006/main',
    'r':   'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
    'wp':  'http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing',
    'xdr': 'http://schemas.openxmlformats.org/drawingml/2006/spreadsheetDrawing',
    'pkg': 'http://schemas.openxmlformats.org/package/2006/relationships',
}

CONTENT_XML_SUFFIXES = ('document.xml', 'presentation.xml', 'workbook.xml', 'sheet', 'slide')
PATH_PATTERN = re.compile(r'(?:[A-Za-z]:\\|\\\\|/home/|/Users/|/root/)[^\s\'"<>&]{5,}', re.IGNORECASE)
USERNAME_PATTERN = re.compile(r'(?:Users|home)[\\/]([^\\/\s]+)', re.IGNORECASE)
EMU_PER_CM = 914400 / 2.54


# ══════════════════════════════════════════════════════════════════════════════
# PARTE 1 — Extracción desde el XML del .docx
# ══════════════════════════════════════════════════════════════════════════════

def _tag(ns: str, local: str) -> str:
    return f'{{{NS[ns]}}}{local}'

def _rels_file_for(xml_path: str) -> str:
    parts = xml_path.rsplit('/', 1)
    return f"{parts[0]}/_rels/{parts[1]}.rels" if len(parts) == 2 else f"_rels/{xml_path}.rels"

def _build_rel_map(z: zipfile.ZipFile) -> dict:
    rel_map = {}
    for fname in z.namelist():
        if not fname.endswith('.rels'):
            continue
        try:
            root = ET.fromstring(z.read(fname))
            entries = {}
            for rel in root.findall(_tag('pkg', 'Relationship')):
                if 'image' not in rel.attrib.get('Type', '').lower():
                    continue
                rid = rel.attrib.get('Id')
                if rid:
                    entries[rid] = {
                        'target':      rel.attrib.get('Target'),
                        'target_mode': rel.attrib.get('TargetMode', 'Internal'),
                    }
            if entries:
                rel_map[fname] = entries
        except ET.ParseError as e:
            print(f"[WARN] {fname}: {e}")
    return rel_map

def _sha256_of_media(z: zipfile.ZipFile, target: str, xml_path: str) -> Optional[str]:
    """Calcula sha256 del archivo de imagen dentro del ZIP para poder hacer join."""
    # target suele ser '../media/image1.png' relativo al xml_path
    base = xml_path.rsplit('/', 1)[0]
    candidates = [
        target,
        os.path.normpath(f"{base}/{target}").replace('\\', '/'),
        f"word/media/{os.path.basename(target)}",
        f"ppt/media/{os.path.basename(target)}",
        f"xl/media/{os.path.basename(target)}",
    ]
    for c in candidates:
        if c in z.namelist():
            try:
                return hashlib.sha256(z.read(c)).hexdigest()
            except:
                pass
    return None


NS['wml'] = 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'


def extract_xml_features(doc_path: str) -> list[dict]:
    doc_name = os.path.basename(doc_path)
    rows = []

    try:
        with zipfile.ZipFile(doc_path, 'r') as z:
            rel_map = _build_rel_map(z)
            files   = z.namelist()

            # Rutas y usernames a nivel documento
            doc_local_paths = []
            doc_usernames   = set()
            for fname in files:
                if not (fname.endswith('.xml') or fname.endswith('.rels')):
                    continue
                try:
                    content = z.read(fname).decode('utf-8', errors='ignore')
                    for p in PATH_PATTERN.findall(content):
                        doc_local_paths.append(p)
                        for u in USERNAME_PATTERN.findall(p):
                            doc_usernames.add(u)
                except:
                    pass

            content_xmls = [f for f in files if f.endswith('.xml')
                            and any(s in f for s in CONTENT_XML_SUFFIXES)]

            for xml_path in content_xmls:
                local_rels = rel_map.get(_rels_file_for(xml_path), {})
                try:
                    content = z.read(xml_path).decode('utf-8', errors='ignore')
                    root    = ET.fromstring(content)
                except (ET.ParseError, KeyError):
                    continue

                # ── El drawing real es {wordprocessingml}drawing ───
                for drawing in root.iter(_tag('wml', 'drawing')):
                    
                    # Puede ser inline o anchor
                    container = (
                        drawing.find(_tag('wp', 'inline')) or
                        drawing.find(_tag('wp', 'anchor'))
                    )
                    if container is None:
                        continue

                    for blip in container.iter(_tag('a', 'blip')):
                        r_embed = blip.attrib.get(_tag('r', 'embed'))
                        r_link  = blip.attrib.get(_tag('r', 'link'))
                        rel_info = local_rels.get(r_embed, {})

                        # docPr y extent dentro del container
                        docpr = container.find(_tag('wp', 'docPr'))
                        ext   = container.find(_tag('wp', 'extent'))

                        img_name  = docpr.attrib.get('name')  if docpr is not None else None
                        img_descr = docpr.attrib.get('descr') if docpr is not None else None
                        img_title = docpr.attrib.get('title') if docpr is not None else None

                        width_emu  = int(ext.attrib['cx']) if ext is not None and 'cx' in ext.attrib else None
                        height_emu = int(ext.attrib['cy']) if ext is not None and 'cy' in ext.attrib else None

                        target     = rel_info.get('target')
                        img_sha256 = _sha256_of_media(z, target, xml_path) if target else None

                        rows.append({
                            'doc_sha256':      doc_name,
                            'img_sha256':      img_sha256,
                            'rel_id':          r_embed,
                            'rel_target':      target,
                            'target_mode':     rel_info.get('target_mode'),
                            'external_link':   r_link,
                            'img_name':        img_name,
                            'img_descr':       img_descr,
                            'img_title':       img_title,
                            'width_cm':        round(width_emu / EMU_PER_CM, 2) if width_emu else None,
                            'height_cm':       round(height_emu / EMU_PER_CM, 2) if height_emu else None,
                            'doc_local_paths': doc_local_paths,
                            'doc_usernames':   list(doc_usernames),
                            'source_xml':      xml_path,
                        })

    except zipfile.BadZipFile:
        print(f"[ERROR] ZIP inválido: {doc_path}")
    except Exception as e:
        print(f"[ERROR] {doc_path}: {e}")

    return rows


# ══════════════════════════════════════════════════════════════════════════════
# PARTE 2 — Extracción desde bytes de la imagen (EXIF, PNG chunks, XMP)
# ══════════════════════════════════════════════════════════════════════════════

EXIF_FIELDS_OF_INTEREST = {
    'Software', 'Artist', 'Copyright', 'Make', 'Model',
    'DateTime', 'DateTimeOriginal', 'DateTimeDigitized',
    'ImageDescription', 'UserComment', 'GPSInfo',
    'ProcessingSoftware', 'HostComputer', 'CameraOwnerName',
    'LensModel', 'DocumentName',
}

def _parse_xmp(data: bytes) -> dict:
    """Extrae campos relevantes del bloque XMP si existe."""
    xmp = {}
    try:
        xmp_start = data.find(b'<x:xmpmeta')
        xmp_end   = data.find(b'</x:xmpmeta>') + len(b'</x:xmpmeta>')
        if xmp_start == -1:
            return xmp
        root = ET.fromstring(data[xmp_start:xmp_end])
        # Campos comunes de autoría
        for tag in root.iter():
            local = tag.tag.split('}')[-1] if '}' in tag.tag else tag.tag
            if local in {'CreatorTool', 'creator', 'Author', 'description',
                         'title', 'subject', 'rights', 'source'}:
                if tag.text and tag.text.strip():
                    xmp[f'xmp_{local}'] = tag.text.strip()
    except:
        pass
    return xmp

def _parse_png_text_chunks(data: bytes) -> dict:
    """Extrae chunks tEXt/iTXt de un PNG (suelen contener Software, Comment, Author...)."""
    texts = {}
    if not data[:8] == b'\x89PNG\r\n\x1a\n':
        return texts
    offset = 8
    while offset < len(data) - 12:
        try:
            length = struct.unpack('>I', data[offset:offset+4])[0]
            chunk_type = data[offset+4:offset+8].decode('ascii', errors='ignore')
            chunk_data = data[offset+8:offset+8+length]
            if chunk_type in ('tEXt', 'iTXt', 'zTXt'):
                null_pos = chunk_data.find(b'\x00')
                if null_pos != -1:
                    key   = chunk_data[:null_pos].decode('latin-1', errors='ignore')
                    value = chunk_data[null_pos+1:].lstrip(b'\x00').decode('utf-8', errors='ignore').strip('\x00')
                    if key and value:
                        texts[f'png_{key}'] = value
            offset += 12 + length
        except:
            break
    return texts

def extract_image_features(raw: bytes) -> dict:
    """
    Extrae features forenses de los bytes de una imagen.
    """
    features = {}

    if not raw:
        return features

    # ── EXIF via Pillow ────────────────────────────────────────────
    try:
        img  = Image.open(io.BytesIO(raw))
        features['img_format'] = img.format
        features['img_mode']   = img.mode
        features['img_width']  = img.width
        features['img_height'] = img.height

        exif_raw = img._getexif() if hasattr(img, '_getexif') else None
        if exif_raw:
            for tag_id, value in exif_raw.items():
                tag_name = TAGS.get(tag_id, str(tag_id))
                if tag_name in EXIF_FIELDS_OF_INTEREST:
                    features[f'exif_{tag_name}'] = str(value)[:200]
    except Exception as e:
        features['img_parse_error'] = str(e)

    # ── PNG text chunks ────────────────────────────────────────────
    features.update(_parse_png_text_chunks(raw))

    # ── XMP metadata ──────────────────────────────────────────────
    features.update(_parse_xmp(raw))

    return features


# ══════════════════════════════════════════════════════════════════════════════
# PARTE 3 — Pipeline completo + join
# ══════════════════════════════════════════════════════════════════════════════

def build_attribution_features(
    doc_dir: str,
    df: pd.DataFrame,   # columnas: doc_sha256, sha256, doc_format, data
) -> pd.DataFrame:

    # ── 1. Features XML — solo docs OOXML ─────────────────────────
    xml_rows = []

    for doc_sha256 in df['doc_sha256'].unique():
        doc_path = os.path.join(doc_dir, doc_sha256)

        if not os.path.exists(doc_path):
            print(f"[SKIP] No existe: {doc_sha256}")
            continue

        if not check_ooxml(doc_path):
            print(f"[SKIP] No es OOXML: {doc_sha256}")
            continue

        rows = extract_xml_features(doc_path)
        xml_rows.extend(rows)
        print(f"[+] {doc_sha256}: {len(rows)} imágenes extraídas")

    df_xml = pd.DataFrame(xml_rows) if xml_rows else pd.DataFrame()
    print(f"\n[+] XML features total: {len(df_xml)} filas")

    # ── 2. Features de bytes de imagen ────────────────────────────
    img_feature_rows = []

    for _, row in df.iterrows():
        raw = row['data']
        if isinstance(raw, str):
            try:
                raw = bytes.fromhex(raw) if not raw.startswith("b'") else \
                      raw[2:-1].encode('raw_unicode_escape').decode('unicode_escape').encode('latin-1')
            except:
                raw = b''

        feats = extract_image_features(raw)
        feats['img_sha256'] = row['sha256']      # sha256 de la imagen
        feats['doc_sha256'] = row['doc_sha256']  # sha256 del documento padre
        img_feature_rows.append(feats)

    df_img = pd.DataFrame(img_feature_rows)
    print(f"[+] Image byte features: {len(df_img)} filas")

    # ── 3. Join ────────────────────────────────────────────────────
    if df_xml.empty:
        print("[WARN] No se extrajeron features XML, devolviendo solo byte features")
        return df_img

    df_out = pd.merge(df_xml, df_img, on=['doc_sha256', 'img_sha256'], how='outer')
    print(f"[+] Features combinados: {df_out.shape}")

    # ── 4. Features derivados ──────────────────────────────────────
    df_out['usernames_str'] = df_out['doc_usernames'].apply(
        lambda x: '|'.join(sorted(set(x))) if isinstance(x, list) else ''
    )
    df_out['img_name_norm'] = (
        df_out['img_name']
        .str.lower()
        .str.replace(r'\d+$', '', regex=True)
        .str.strip()
    )
    df_out['software'] = (
        df_out.get('exif_Software', pd.Series(dtype=str))
              .fillna(df_out.get('xmp_CreatorTool', pd.Series(dtype=str)))
    )

    return df_out


# ── Uso ───────────────────────────────────────────────────────────────────────

'''df_features = build_attribution_features(
    doc_dir   = '/home/jupyter/binaries/',
    df = df_imagedata,   # tu DataFrame original
)

# Vista rápida de columnas con más señal para atribución
attribution_cols = [
    'doc_sha256', 'img_sha256',
    'usernames_str', 'img_name', 'img_name_norm',
    'img_descr', 'img_title', 'doc_local_paths',
    'software', 'exif_Artist', 'exif_Copyright',
    'xmp_CreatorTool', 'xmp_creator',
    'target_mode', 'external_link',
]
print(df_features[[c for c in attribution_cols if c in df_features.columns]].head(20))'''

In [ ]:
rows = extract_xml_features('/home/jupyter/binaries/' + df_imagedata['doc_sha256'].iloc[0])
print(f"Filas: {len(rows)}")
for r in rows:
    print(r['img_name'], r['img_descr'], r['rel_target'], r['img_sha256'])

In [ ]:
df_features = build_attribution_features(
    doc_dir = '/home/jupyter/binaries/',
    df      = df_imagedata,
)

# Ver señales de atribución
attribution_cols = [
    'doc_sha256', 'img_sha256',
    'img_name', 'img_descr', 'img_title',
    'usernames_str', 'doc_local_paths',
    'software', 'target_mode', 'external_link',
    'width_cm', 'height_cm',
]
print(df_features[[c for c in attribution_cols if c in df_features.columns]])

# Resumen rápido de señales
print("\n=== Idiomas detectados en img_name ===")
print(df_features['img_name'].dropna().unique())

print("\n=== Usernames encontrados ===")
all_users = df_features['usernames_str'].str.split('|').explode().dropna()
print(all_users[all_users != ''].value_counts())

print("\n=== Rutas locales únicas ===")
all_paths = df_features['doc_local_paths'].explode().dropna()
print(all_paths.value_counts().head(20))

In [ ]:
df_features

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from scipy.sparse import hstack

# ── 1. Agregar features a nivel documento ─────────────────────────
doc_features = df_features.groupby('doc_sha256').agg(
    # Idioma dominante del sistema
    img_names_all   = ('img_name', lambda x: ' '.join(x.dropna())),
    # Usernames
    usernames       = ('usernames_str', 'first'),
    # Software
    software_all    = ('software', lambda x: ' '.join(x.dropna())),
    # Rutas — concatenar todas
    paths_all       = ('doc_local_paths', lambda x: ' '.join(
                            [p for sublist in x if isinstance(sublist, list) 
                             for p in sublist])),
    # Dimensiones medianas (estilo de maquetación)
    width_median    = ('width_cm',  'median'),
    height_median   = ('height_cm', 'median'),
    n_images        = ('img_sha256', 'count'),
).reset_index()

# ── 2. Detectar idioma del sistema por img_name ────────────────────
def detect_locale(names: str) -> str:
    if any(c in names for c in 'РрисунокНадпись'):
        return 'ru'
    if any('\uAC00' <= c <= '\uD7A3' for c in names):  # Hangul
        return 'ko'
    if any('\u4E00' <= c <= '\u9FFF' for c in names):  # CJK
        return 'zh'
    if any('\u3040' <= c <= '\u30FF' for c in names):  # Hiragana/Katakana
        return 'ja'
    if any('\u0600' <= c <= '\u06FF' for c in names):  # Árabe
        return 'ar'
    if 'Picture' in names or 'Image' in names:
        return 'en'
    return 'unknown'

doc_features['locale'] = doc_features['img_names_all'].apply(detect_locale)

# ── 3. Feature matrix para clustering ─────────────────────────────

# TF-IDF sobre rutas (captura patrones de nomenclatura y estructura)
tfidf_paths = TfidfVectorizer(
    analyzer    = 'char_wb',
    ngram_range = (3, 5),      # trigramas de caracteres → robusto a variaciones
    max_features = 500,
    min_df      = 2,
).fit_transform(doc_features['paths_all'].fillna(''))

# TF-IDF sobre nombres de imagen
tfidf_names = TfidfVectorizer(
    analyzer    = 'char_wb',
    ngram_range = (2, 4),
    max_features = 200,
    min_df      = 2,
).fit_transform(doc_features['img_names_all'].fillna(''))

# TF-IDF sobre software
tfidf_soft = TfidfVectorizer(
    max_features = 100,
    min_df       = 2,
).fit_transform(doc_features['software_all'].fillna(''))

# Features numéricas
import pandas as pd
num_features = doc_features[['width_median', 'height_median', 'n_images']].fillna(0)
from scipy.sparse import csr_matrix
num_sparse = csr_matrix(num_features.values)

# Combinar todo
X = hstack([tfidf_paths, tfidf_names, tfidf_soft, num_sparse])

# ── 4. DBSCAN ─────────────────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_distances

D = cosine_distances(X)

db = DBSCAN(
    metric      = 'precomputed',
    eps         = 0.3,    # Densidad
    min_samples = 2,
).fit(D)

doc_features['cluster'] = db.labels_

# ── 5. Resumen de clusters ─────────────────────────────────────────
print("=== Clusters ===")
print(doc_features['cluster'].value_counts())

print("\n=== Detalle por cluster ===")
for cluster_id in sorted(doc_features['cluster'].unique()):
    if cluster_id == -1:
        continue  # ruido
    group = doc_features[doc_features['cluster'] == cluster_id]
    print(f"\n── Cluster {cluster_id} ({len(group)} docs) ──")
    print(f"  Locale:    {group['locale'].value_counts().to_dict()}")
    print(f"  Usernames: {group['usernames'].unique()}")
    print(f"  Software:  {group['software_all'].unique()[:3]}")
    print(f"  Rutas rep: {group['paths_all'].iloc[0][:200]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Ver distribución de distancias para elegir eps ────────────────
# Distancia al k-ésimo vecino más cercano (k = min_samples)
k = 2
kth_distances = np.sort(D, axis=1)[:, k]  # distancia al 2º vecino

plt.figure(figsize=(10, 4))
plt.plot(np.sort(kth_distances))
plt.axhline(y=0.3, color='r', linestyle='--', label='eps actual (0.3)')
plt.xlabel('Documentos (ordenados)')
plt.ylabel(f'Distancia al vecino {k}')
plt.title('K-distance plot — buscar el "codo" para eps')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Percentiles de distancia al vecino {k}:")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{p}: {np.percentile(kth_distances, p):.4f}")

# ── Diagnóstico: ver similitud entre docs conocidos ───────────────
# Compara docs del mismo username — deberían estar cerca
idx_911   = doc_features[doc_features['usernames'].str.contains('911',   na=False)].index.tolist()
idx_julien= doc_features[doc_features['usernames'].str.contains('Julien',na=False)].index.tolist()
idx_pc    = doc_features[doc_features['usernames'].str.contains('^pc$',  na=False, regex=True)].index.tolist()

print(f"\nDistancias internas cluster '911' (deberían ser bajas):")
if len(idx_911) > 1:
    sub = D[np.ix_(idx_911, idx_911)]
    print(f"  media={sub.mean():.4f}  max={sub.max():.4f}")

print(f"Distancias internas cluster 'Julien':")
if len(idx_julien) > 1:
    sub = D[np.ix_(idx_julien, idx_julien)]
    print(f"  media={sub.mean():.4f}  max={sub.max():.4f}")

print(f"\nDistancia entre '911' y 'Julien' (deberían ser altas):")
if idx_911 and idx_julien:
    sub = D[np.ix_(idx_911, idx_julien)]
    print(f"  media={sub.mean():.4f}  min={sub.min():.4f}")

In [ ]:
# ── Diagnóstico: cuántos docs tienen features reales ──────────────
print("Docs con rutas:    ", (doc_features['paths_all'] != '').sum())
print("Docs con usernames:", (doc_features['usernames'] != '').sum())
print("Docs con software: ", (doc_features['software_all'] != '').sum())
print("Docs con locale:   ", (doc_features['locale'] != 'unknown').sum())
print("Docs con XML:      ", doc_features['img_names_all'].str.strip().ne('').sum())
print("\nTotal docs:", len(doc_features))

# ── Ver qué tienen los docs sin ninguna feature ───────────────────
sin_features = doc_features[
    (doc_features['paths_all'] == '') &
    (doc_features['usernames'] == '') &
    (doc_features['software_all'] == '') &
    (doc_features['locale'] == 'unknown')
]
print(f"\nDocs sin ninguna feature forense: {len(sin_features)}")
print(sin_features[['doc_sha256', 'n_images', 'width_median', 'height_median']].head(10))

In [ ]:
# ── Clustering solo sobre docs con alguna feature ─────────────────
mask = (
    (doc_features['paths_all'] != '') |
    (doc_features['usernames'] != '') |
    (doc_features['software_all'] != '') |
    (doc_features['locale'] != 'unknown')
)
df_rich = doc_features[mask].reset_index(drop=True)
print(f"Docs con features: {len(df_rich)}")

# Reconstruir X solo con estos docs
tfidf_paths = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=500, min_df=2)
tfidf_names = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=200, min_df=2)
tfidf_soft  = TfidfVectorizer(max_features=100, min_df=2)

X_paths = tfidf_paths.fit_transform(df_rich['paths_all'].fillna(''))
X_names = tfidf_names.fit_transform(df_rich['img_names_all'].fillna(''))
X_soft  = tfidf_soft.fit_transform(df_rich['software_all'].fillna(''))
X_num   = csr_matrix(df_rich[['width_median','height_median','n_images']].fillna(0).values)

X = hstack([X_paths, X_names, X_soft, X_num])
D = cosine_distances(X)

# eps basado en percentiles reales de este subconjunto
k = 2
kth = np.sort(D, axis=1)[:, k]
print("\nPercentiles distancia al vecino 2:")
for p in [50, 75, 90, 95, 99]:
    print(f"  p{p}: {np.percentile(kth, p):.4f}")

# Probar varios eps
for eps in [0.005, 0.01, 0.02, 0.05, 0.10]:
    db = DBSCAN(metric='precomputed', eps=eps, min_samples=2).fit(D)
    labels = db.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    print(f"eps={eps:.3f} → {n_clusters} clusters, {n_noise} ruido ({n_noise/len(df_rich)*100:.0f}%)")

In [ ]:
# ── Clustering final ──────────────────────────────────────────────
db = DBSCAN(metric='precomputed', eps=0.01, min_samples=2).fit(D)
df_rich['cluster'] = db.labels_

# ── Detalle por cluster ───────────────────────────────────────────
for cid in sorted(df_rich['cluster'].unique()):
    group = df_rich[df_rich['cluster'] == cid]
    label = f"RUIDO ({len(group)} docs)" if cid == -1 else f"Cluster {cid} ({len(group)} docs)"
    print(f"\n{'═'*55}")
    print(f"  {label}")
    print(f"{'═'*55}")
    print(f"  Locale:    {group['locale'].value_counts().to_dict()}")
    
    users = group['usernames'].str.split('|').explode()
    users = users[users != ''].value_counts()
    if not users.empty:
        print(f"  Usernames: {users.to_dict()}")
    
    soft = group['software_all'][group['software_all'] != ''].unique()
    if len(soft):
        print(f"  Software:  {list(soft[:5])}")
    
    # Rutas más frecuentes dentro del cluster
    all_paths = group['paths_all'].str.split().explode().dropna()
    all_paths = all_paths[all_paths != '']
    if not all_paths.empty:
        print(f"  Rutas top: {all_paths.value_counts().head(3).to_dict()}")
    
    print(f"  n_images:  min={group['n_images'].min()}  "
          f"max={group['n_images'].max()}  "
          f"median={group['n_images'].median()}")
    print(f"  Doc IDs:   {list(group['doc_sha256'].str[:16])}")

In [ ]:
# ── Sub-clustering del cluster 2 ──────────────────────────────────
df_c2 = df_rich[df_rich['cluster'] == 2].reset_index(drop=True)

X2_paths = tfidf_paths.transform(df_c2['paths_all'].fillna(''))
X2_names = tfidf_names.transform(df_c2['img_names_all'].fillna(''))
X2_soft  = tfidf_soft.transform(df_c2['software_all'].fillna(''))
X2_num   = csr_matrix(df_c2[['width_median','height_median','n_images']].fillna(0).values)

X2 = hstack([X2_paths, X2_names, X2_soft, X2_num])
D2 = cosine_distances(X2)

for eps in [0.001, 0.005, 0.01, 0.02]:
    db2 = DBSCAN(metric='precomputed', eps=eps, min_samples=2).fit(D2)
    n_c = len(set(db2.labels_)) - (1 if -1 in db2.labels_ else 0)
    n_n = (db2.labels_ == -1).sum()
    print(f"eps={eps:.3f} → {n_c} clusters, {n_n} ruido")

# Con el eps que tenga más sentido:
db2 = DBSCAN(metric='precomputed', eps=0.005, min_samples=2).fit(D2)
df_c2['subcluster'] = db2.labels_

for cid in sorted(df_c2['subcluster'].unique()):
    g = df_c2[df_c2['subcluster'] == cid]
    label = 'RUIDO' if cid == -1 else f'Sub-cluster 2.{cid}'
    print(f"\n── {label} ({len(g)} docs) ──")
    print(f"  Locale:    {g['locale'].value_counts().to_dict()}")
    users = g['usernames'].str.split('|').explode()
    users = users[users != ''].value_counts()
    if not users.empty:
        print(f"  Usernames: {users.to_dict()}")
    soft = g['software_all'][g['software_all'] != ''].unique()
    if len(soft):
        print(f"  Software:  {list(soft[:3])}")
    paths = g['paths_all'].str.split().explode().dropna()
    paths = paths[paths != '']
    if not paths.empty:
        print(f"  Rutas top: {paths.value_counts().head(3).to_dict()}")
    print(f"  Docs:      {list(g['doc_sha256'].str[:16])}")

# METADATA

Lectura de propiedades de documento y combinación de señales de autoría.

## OOXML

In [ ]:
# ─────────────────────────────────────────────
# OOXML (docx, xlsx, pptx)
# ─────────────────────────────────────────────

def extract_ooxml_metadata(path):
    """
    Extrae todos los metadatos posibles de un archivo OOXML.
    Cubre: core properties, app properties, custom properties,
    settings, theme, y archivos .rels
    """
    meta = {"sha256": os.path.basename(path), "format": "ooxml"}

    try:
        with zipfile.ZipFile(path, 'r') as z:
            files = z.namelist()

            # ── 1. Core Properties (docProps/core.xml) ──────────────────
            # Autor, fechas, título, descripción, idioma, revisión
            if 'docProps/core.xml' in files:
                root = ET.fromstring(z.read('docProps/core.xml'))
                ns = {
                    'dc':      'http://purl.org/dc/elements/1.1/',
                    'cp':      'http://schemas.openxmlformats.org/package/2006/metadata/core-properties',
                    'dcterms': 'http://purl.org/dc/terms/',
                    'xsi':     'http://www.w3.org/2001/XMLSchema-instance'
                }
                def get(tag, ns_key):
                    el = root.find(f'{{{ns[ns_key]}}}{tag}')
                    return el.text if el is not None else None

                meta['title']          = get('title', 'dc')
                meta['subject']        = get('subject', 'dc')
                meta['creator']        = get('creator', 'dc')          # autor original
                meta['description']    = get('description', 'dc')
                meta['keywords']       = get('keywords', 'cp')
                meta['last_modified_by'] = get('lastModifiedBy', 'cp') # último editor
                meta['revision']       = get('revision', 'cp')
                meta['language']       = get('language', 'dc')
                meta['category']       = get('category', 'cp')
                meta['content_status'] = get('contentStatus', 'cp')
                meta['created']        = get('created', 'dcterms')
                meta['modified']       = get('modified', 'dcterms')

            # ── 2. App Properties (docProps/app.xml) ─────────────────────
            # Aplicación, versión, empresa, template, páginas, palabras
            if 'docProps/app.xml' in files:
                root = ET.fromstring(z.read('docProps/app.xml'))
                ns_a = 'http://schemas.openxmlformats.org/officeDocument/2006/extended-properties'
                def getapp(tag):
                    el = root.find(f'{{{ns_a}}}{tag}')
                    return el.text if el is not None else None

                meta['application']       = getapp('Application')    # ej: "Microsoft Office Word"
                meta['app_version']       = getapp('AppVersion')     # ej: "16.0000"
                meta['company']           = getapp('Company')        # empresa del autor
                meta['template']          = getapp('Template')       # plantilla usada
                meta['manager']           = getapp('Manager')
                meta['pages']             = getapp('Pages')
                meta['words']             = getapp('Words')
                meta['characters']        = getapp('Characters')
                meta['doc_security']      = getapp('DocSecurity')    # 0=sin protección
                meta['hyperlinks_changed']= getapp('HyperlinksChanged')
                meta['shared_doc']        = getapp('SharedDoc')
                meta['links_up_to_date']  = getapp('LinksUpToDate')
                meta['scale_crop']        = getapp('ScaleCrop')
                meta['total_time']        = getapp('TotalTime')      # tiempo de edición (minutos)

            # ── 3. Custom Properties (docProps/custom.xml) ───────────────
            # Propiedades personalizadas — muy raras pero muy indicativas
            if 'docProps/custom.xml' in files:
                root = ET.fromstring(z.read('docProps/custom.xml'))
                ns_c = 'http://schemas.openxmlformats.org/officeDocument/2006/custom-properties'
                ns_vt = 'http://schemas.openxmlformats.org/officeDocument/2006/docPropsVTypes'
                custom = {}
                for prop in root.findall(f'{{{ns_c}}}property'):
                    name = prop.attrib.get('name')
                    for child in prop:
                        custom[name] = child.text
                meta['custom_properties'] = custom

            # ── 4. Settings (word/settings.xml o equivalente) ────────────
            # RSIDs, idioma, template adjunto, compatibilidad
            settings_paths = ['word/settings.xml', 'xl/workbook.xml', 'ppt/presentation.xml']
            for sp in settings_paths:
                if sp in files:
                    root = ET.fromstring(z.read(sp))
                    ns_w = 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'

                    # Template adjunto (ruta → puede revelar origen)
                    attached = root.find(f'{{{ns_w}}}attachedTemplate')
                    if attached is not None:
                        meta['attached_template_id'] = attached.attrib.get(
                            '{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id'
                        )

                    # Idioma por defecto del documento
                    lang = root.find(f'.//{{{ns_w}}}lang')
                    if lang is not None:
                        meta['default_lang_val'] = lang.attrib.get(f'{{{ns_w}}}val')
                        meta['default_lang_eastAsia'] = lang.attrib.get(f'{{{ns_w}}}eastAsia')
                        meta['default_lang_bidi'] = lang.attrib.get(f'{{{ns_w}}}bidi')

                    # RSIDs (todos los que aparezcan)
                    rsids = [el.attrib.get(f'{{{ns_w}}}val')
                             for el in root.iter(f'{{{ns_w}}}rsid')
                             if el.attrib.get(f'{{{ns_w}}}val')]
                    if rsids:
                        meta['rsids'] = list(set(rsids))
                    break

            # ── 5. Relaciones externas (.rels) ────────────────────────────
            # External links, templates remotos, OLE objects
            external_rels = []
            for fname in files:
                if fname.endswith('.rels'):
                    try:
                        root = ET.fromstring(z.read(fname))
                        ns_r = 'http://schemas.openxmlformats.org/package/2006/relationships'
                        for rel in root.findall(f'{{{ns_r}}}Relationship'):
                            target_mode = rel.attrib.get('TargetMode', '')
                            if target_mode == 'External':
                                external_rels.append({
                                    'type':   rel.attrib.get('Type', '').split('/')[-1],
                                    'target': rel.attrib.get('Target'),
                                    'source_file': fname
                                })
                    except Exception:
                        pass
            if external_rels:
                meta['external_relationships'] = external_rels

            # ── 6. Theme (colores, fuentes del tema) ─────────────────────
            theme_paths = [f for f in files if 'theme/theme' in f and f.endswith('.xml')]
            if theme_paths:
                try:
                    root = ET.fromstring(z.read(theme_paths[0]))
                    ns_a = 'http://schemas.openxmlformats.org/drawingml/2006/main'
                    # Fuentes principales del tema
                    latin = root.find(f'.//{{{ns_a}}}latin')
                    ea    = root.find(f'.//{{{ns_a}}}ea')
                    cs    = root.find(f'.//{{{ns_a}}}cs')
                    meta['theme_font_latin'] = latin.attrib.get('typeface') if latin is not None else None
                    meta['theme_font_ea']    = ea.attrib.get('typeface')    if ea    is not None else None
                    meta['theme_font_cs']    = cs.attrib.get('typeface')    if cs    is not None else None
                    meta['theme_name']       = root.attrib.get('name')
                except Exception:
                    pass

    except Exception as e:
        meta['error'] = str(e)

    return meta


# ─────────────────────────────────────────────
# OLE2 (doc, xls, ppt)
# ─────────────────────────────────────────────

def extract_ole2_metadata(path):
    """
    Extrae todos los metadatos posibles de un archivo OLE2.
    Cubre: SummaryInformation, DocumentSummaryInformation,
    streams disponibles (CLSIDs, timestamps, etc.)
    """
    meta = {"sha256": os.path.basename(path), "format": "ole2"}

    try:
        ole = olefile.OleFileIO(path)

        # ── 1. SummaryInformation ─────────────────────────────────────
        # Autor, título, fechas, aplicación, seguridad...
        if ole.exists('\x05SummaryInformation'):
            si = ole.get_metadata()

            def decode(val):
                if isinstance(val, bytes):
                    for enc in ('utf-8', 'latin-1', 'cp1252'):
                        try: return val.decode(enc)
                        except: pass
                    return val.hex()
                if isinstance(val, datetime):
                    return val.isoformat()
                return val

            meta['title']          = decode(si.title)
            meta['subject']        = decode(si.subject)
            meta['creator']        = decode(si.author)           # autor original
            meta['keywords']       = decode(si.keywords)
            meta['description']    = decode(si.comments)
            meta['last_modified_by'] = decode(si.last_saved_by) # último editor
            meta['revision']       = decode(si.revision_number)
            meta['application']    = decode(si.creating_application)
            meta['created']        = decode(si.create_time)
            meta['modified']       = decode(si.last_saved_time)
            meta['total_time']     = decode(si.total_edit_time)
            meta['pages']          = si.num_pages
            meta['words']          = si.num_words
            meta['characters']     = si.num_chars
            meta['doc_security']   = si.doc_security
            meta['thumbnail']      = True if si.thumbnail else False

        # ── 2. DocumentSummaryInformation ────────────────────────────
        # Empresa, manager, categoría, versión, idioma...
        if ole.exists('\x05DocumentSummaryInformation'):
            dsi = ole.get_metadata()
            def decode(val):
                if isinstance(val, bytes):
                    for enc in ('utf-8', 'latin-1', 'cp1252'):
                        try: return val.decode(enc)
                        except: pass
                    return val.hex()
                return val

            meta['company']        = decode(dsi.company)
            meta['manager']        = decode(dsi.manager)
            meta['category']       = decode(dsi.category)
            meta['app_version']    = dsi.app_version
            meta['language']       = dsi.language
            meta['content_status'] = decode(dsi.content_status)
            meta['content_type']   = decode(dsi.content_type)

        # ── 3. CLSID del root entry ───────────────────────────────────
        # Identifica la aplicación que creó el archivo
        root_entry = ole.root
        if root_entry.clsid:
            meta['root_clsid'] = root_entry.clsid

        # ── 4. Todos los streams disponibles ─────────────────────────
        # Lista completa de streams → revela estructura interna
        meta['streams'] = ole.listdir(streams=True, storages=False)
        meta['storages'] = ole.listdir(streams=False, storages=True)

        # ── 5. CLSIDs de todos los entries ────────────────────────────
        clsids = {}
        for entry in ole.direntries:
            if entry is not None and entry.clsid and entry.clsid != '00000000-0000-0000-0000-000000000000':
                clsids[entry.name] = entry.clsid
        if clsids:
            meta['clsids'] = clsids

        # ── 6. Timestamps de entries ──────────────────────────────────
        timestamps = {}
        for entry in ole.direntries:
            if entry is not None:
                if entry.modify_time:
                    timestamps[entry.name] = {
                        'created': entry.create_time.isoformat() if entry.create_time else None,
                        'modified': entry.modify_time.isoformat() if entry.modify_time else None,
                    }
        if timestamps:
            meta['entry_timestamps'] = timestamps

        # ── 7. WordDocument stream (solo .doc) ───────────────────────
        # Extrae flags del FIB (File Information Block)
        if ole.exists('WordDocument'):
            stream = ole.openstream('WordDocument').read()
            if len(stream) > 32:
                # wIdent: magic number (debe ser 0xA5EC para .doc)
                w_ident = struct.unpack_from('<H', stream, 0)[0]
                # nFib: versión del formato
                n_fib   = struct.unpack_from('<H', stream, 2)[0]
                # fEncrypted flag (byte 11, bit 0)
                f_encrypted = bool(stream[11] & 0x01)
                meta['doc_magic']     = hex(w_ident)
                meta['doc_fib_version'] = n_fib
                meta['doc_encrypted'] = f_encrypted

        ole.close()

    except Exception as e:
        meta['error'] = str(e)

    return meta


# ─────────────────────────────────────────────
# Dispatcher unificado
# ─────────────────────────────────────────────


def extract_metadata(path):
    if check_ooxml(path):
        return extract_ooxml_metadata(path)
    elif check_ole2(path):
        return extract_ole2_metadata(path)
    else:
        # Intentar detectar por magic bytes
        with open(path, 'rb') as f:
            magic = f.read(4)
        if magic == b'PK\x03\x04':
            return extract_ooxml_metadata(path)
        elif magic == b'\xd0\xcf\x11\xe0':
            return extract_ole2_metadata(path)
        return {"sha256": os.path.basename(path), "error": "formato no reconocido"}

## OLE2

In [ ]:
import olefile
import struct
import os

# ── Mapas de Property IDs ──────────────────────────────────────────

# \x05SummaryInformation (FMTID: {F29F85E0-4FF9-1068-AB91-08002B27B3D9})
SUMMARY_PROPS = {
    0x02: 'title',
    0x03: 'subject',
    0x04: 'creator',          # author original  ← equivale a dc:creator
    0x05: 'keywords',
    0x06: 'description',
    0x07: 'template',
    0x08: 'last_modified_by', # last author      ← equivale a cp:lastModifiedBy
    0x09: 'revision',         #                  ← equivale a cp:revision
    0x0A: 'total_time',       # edit time en 100-ns intervals ← cp:totalTime
    0x0B: 'last_printed',
    0x0C: 'created',
    0x0D: 'modified',
    0x0E: 'pages',            #                  ← equivale a Pages en app.xml
    0x0F: 'words',            #                  ← equivale a Words
    0x10: 'characters',       #                  ← equivale a Characters
    0x11: 'thumbnail',
    0x12: 'application',      # creating app     ← equivale a Application
    0x13: 'doc_security',     #                  ← equivale a DocSecurity
    0x80000000: 'locale',  # PID_LOCALE — LCID del property set
}

# \x05DocumentSummaryInformation (FMTID: {D5CDD502-2E9C-101B-9397-08002B2CF9AE})
DOCSUMMARY_PROPS = {
    0x02: 'category',         #                  ← equivale a cp:category
    0x03: 'presentation_format',
    0x04: 'byte_count',
    0x05: 'line_count',
    0x06: 'paragraph_count',
    0x07: 'slide_count',
    0x08: 'note_count',
    0x09: 'hidden_slide_count',
    0x0A: 'mmclip_count',
    0x0B: 'scale_crop',       #                  ← equivale a ScaleCrop
    0x0C: 'heading_pairs',
    0x0D: 'titles_of_parts',
    0x0E: 'manager',          #                  ← equivale a Manager
    0x0F: 'company',          #                  ← equivale a Company
    0x10: 'links_up_to_date', #                  ← equivale a LinksUpToDate
    0x11: 'characters_with_spaces',
    0x13: 'shared_doc',       #                  ← equivale a SharedDoc
    0x16: 'hyperlinks_changed',#                 ← equivale a HyperlinksChanged
    0x17: 'app_version',      # ej: 0x000F0000 = 15.0 ← equivale a AppVersion
    # 0x18 en adelante: user-defined properties (segunda sección)
}

# VT types del PROPSET
VT_EMPTY       = 0x0000
VT_I2          = 0x0002
VT_I4          = 0x0003
VT_R4          = 0x0004
VT_R8          = 0x0005
VT_BOOL        = 0x000B
VT_BSTR        = 0x0008
VT_LPSTR       = 0x001E
VT_LPWSTR      = 0x001F
VT_FILETIME    = 0x0040
VT_BLOB        = 0x0041
VT_CF          = 0x0047
VT_VECTOR      = 0x1000



CODEPAGE_TO_PYTHON = {
    1252: 'cp1252',   # Western European
    1251: 'cp1251',   # Cyrillic
    1250: 'cp1250',   # Central European
    1256: 'cp1256',   # Arabic
    1255: 'cp1255',   # Hebrew
    1253: 'cp1253',   # Greek
    932:  'cp932',    # Japanese
    936:  'gbk',      # Chinese Simplified
    950:  'cp950',    # Chinese Traditional
    949:  'cp949',    # Korean
    65001: 'utf-8',
    0:    'cp1252',   # default
}




def _parse_lpstr(data, offset):
    size = struct.unpack_from('<I', data, offset)[0]
    raw = data[offset+4 : offset+4+size]
    text = raw.rstrip(b'\x00')
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            return text.decode(enc), offset + 4 + ((size + 3) & ~3)
        except:
            continue
    return raw.hex(), offset + 4 + ((size + 3) & ~3)


def _parse_lpwstr(data, offset):
    size = struct.unpack_from('<I', data, offset)[0]  # en caracteres
    raw = data[offset+4 : offset+4+size*2]
    text = raw.decode('utf-16-le', errors='replace').rstrip('\x00')
    return text, offset + 4 + ((size * 2 + 3) & ~3)


def _parse_filetime(data, offset):
    """FILETIME de Windows → ISO string."""
    lo, hi = struct.unpack_from('<II', data, offset)
    ft = (hi << 32) | lo
    if ft == 0:
        return None
    try:
        from datetime import datetime, timedelta, timezone
        EPOCH = datetime(1601, 1, 1, tzinfo=timezone.utc)
        dt = EPOCH + timedelta(microseconds=ft // 10)
        return dt.isoformat()
    except:
        return None


def _parse_app_version(raw_int):
    """0x000F0000 → '15.0'"""
    major = (raw_int >> 16) & 0xFFFF
    minor = raw_int & 0xFFFF
    return f"{major}.{minor}"


def parse_property_set(data, prop_map):
    result = {}
    try:
        num_sets  = struct.unpack_from('<I', data, 24)[0]
        set_offset = struct.unpack_from('<I', data, 44)[0]
        num_props  = struct.unpack_from('<I', data, set_offset + 4)[0]

        # ── Primer paso: buscar codepage (PID=1) antes de parsear strings ──
        codepage = 1252  # default
        for i in range(num_props):
            entry_offset = set_offset + 8 + i * 8
            prop_id  = struct.unpack_from('<I', data, entry_offset)[0]
            prop_off = struct.unpack_from('<I', data, entry_offset + 4)[0]
            abs_off  = set_offset + prop_off
            vt = struct.unpack_from('<H', data, abs_off)[0]
            if prop_id == 0x0001 and vt == VT_I2:  # PID_CODEPAGE
                codepage = struct.unpack_from('<h', data, abs_off + 4)[0] & 0xFFFF
                break

        encoding = CODEPAGE_TO_PYTHON.get(codepage, 'cp1252')

        # ── Segundo paso: parsear todos los props con el encoding correcto ──
        for i in range(num_props):
            entry_offset = set_offset + 8 + i * 8
            prop_id  = struct.unpack_from('<I', data, entry_offset)[0]
            prop_off = struct.unpack_from('<I', data, entry_offset + 4)[0]
            abs_off  = set_offset + prop_off
            vt = struct.unpack_from('<H', data, abs_off)[0]
            val_off  = abs_off + 4
            name = prop_map.get(prop_id)

            try:
                if vt == VT_LPSTR:
                    # Usar el encoding detectado del PROPSET
                    size = struct.unpack_from('<I', data, val_off)[0]
                    raw  = data[val_off+4 : val_off+4+size].rstrip(b'\x00')
                    val  = raw.decode(encoding, errors='replace')
                elif vt == VT_LPWSTR:
                    val, _ = _parse_lpwstr(data, val_off)  # UTF-16LE, siempre ok
                elif vt == VT_I2:
                    val = struct.unpack_from('<h', data, val_off)[0]
                elif vt == VT_I4:
                    raw_int = struct.unpack_from('<i', data, val_off)[0]
                    val = _parse_app_version(raw_int) if name == 'app_version' else raw_int
                elif vt == VT_BOOL:
                    val = bool(struct.unpack_from('<H', data, val_off)[0])
                elif vt == VT_FILETIME:
                    val = _parse_filetime(data, val_off)
                else:
                    val = None
            except Exception:
                val = None

            if name and val is not None:
                result[name] = val
            elif name is None and prop_id not in (0x00, 0x01):
                result[f'_unknown_0x{prop_id:08X}'] = val

        result['_codepage'] = codepage
        result['_encoding'] = encoding

    except Exception as e:
        result['_parse_error'] = str(e)

    return result


def extract_ole2_metadata(path):
    """Extrae todos los campos equivalentes a los de OOXML."""
    meta = {}
    meta['sha256'] = os.path.basename(path)

    try:
        with olefile.OleFileIO(path) as ole:

            # ── 1. \x05SummaryInformation ──────────────────────────
            if ole.exists('\x05SummaryInformation'):
                raw = ole.openstream('\x05SummaryInformation').read()
                meta.update(parse_property_set(raw, SUMMARY_PROPS))

            # ── 2. \x05DocumentSummaryInformation ──────────────────
            if ole.exists('\x05DocumentSummaryInformation'):
                raw = ole.openstream('\x05DocumentSummaryInformation').read()
                meta.update(parse_property_set(raw, DOCSUMMARY_PROPS))

            # ── 3. app_version: formatear si llegó como int crudo ──
            if 'app_version' in meta and isinstance(meta['app_version'], int):
                meta['app_version'] = _parse_app_version(meta['app_version'])

            # ── 4. total_time: convertir a minutos legibles ─────────
            if 'total_time' in meta:
                meta['total_time_min'] = meta.pop('total_time')

            # ── 5. language como nombre ───────────────────────────────
            # (SummaryInfo no tiene language directamente;
            #  está en el stream Word Document en los párrafos)
            # Para OLE2 puro se queda como codepage:
            meta['codepage'] = ole.get_metadata().codepage

    except Exception as e:
        meta['_error'] = str(e)

    return meta


# ── Ejecución ───────────────────────────────────────────────────────

list_to_df = []

for docname in csv_full['sha256']:
    path = f"/home/jupyter/binaries/{docname}"
    try:
        if os.path.exists(path) and check_ole2(path):
            row = extract_ole2_metadata(path)
            list_to_df.append(row)
    except Exception as e:
        print(f"Error procesando {docname}: {e}")

df_meta_ole2 = pd.DataFrame(list_to_df)

In [ ]:
df_meta_ole2

In [ ]:
# ver la columna template 
print(df_meta_ole2['last_modified_by'].value_counts(dropna=False))

## Procesamiento, analisis, union... de los DF

In [ ]:
# Valores distintos en 'creator' y 'last_modified_by'
print("OOXML - creator:", df_meta_ooxml['creator'].nunique())
print("OOXML - last_modified_by:", df_meta_ooxml['last_modified_by'].nunique())
print("OLE2 - creator:", df_meta_ole2['creator'].nunique())
print("OLE2 - last_modified_by:", df_meta_ole2['last_modified_by'].nunique())


# crear un diccionario inverso para, por cada valor distinto de 'creator' y 'last_modified_by' de cada df, ver cuántos documentos distintos hay
creator_dict_ooxml = defaultdict(set)
last_modified_by_dict_ooxml = defaultdict(set)
no_same_doc_apperances = defaultdict(set)

creator_dict_ole2 = defaultdict(set)
last_modified_by_dict_ole2 = defaultdict(set)

for index, row in df_meta_ooxml.iterrows():
    creator_dict_ooxml[row['creator']].add(row['sha256'])
    last_modified_by_dict_ooxml[row['last_modified_by']].add(row['sha256'])

for index, row in df_meta_ole2.iterrows():
    creator_dict_ole2[row['creator']].add(row['sha256'])
    last_modified_by_dict_ole2[row['last_modified_by']].add(row['sha256'])


# ahora, por cada diccionario, crear un counter para ver cuantas veces aparece cada valor distinto de 'creator' y 'last_modified_by' en cada df
creator_counter_ooxml = Counter({k: len(v) for k, v in creator_dict_ooxml.items()})
last_modified_by_counter_ooxml = Counter({k: len(v) for k, v in last_modified_by_dict_ooxml.items()})
creator_counter_ole2 = Counter({k: len(v) for k, v in creator_dict_ole2.items()})
last_modified_by_counter_ole2 = Counter({k: len(v) for k, v in last_modified_by_dict_ole2.items()})

print("OOXML - creator counter:", creator_counter_ooxml.most_common(10))
print("OOXML - last_modified_by counter:", last_modified_by_counter_ooxml.most_common(10))
print("OLE2 - creator counter:", creator_counter_ole2.most_common(10))
print("OLE2 - last_modified_by counter:", last_modified_by_counter_ole2.most_common(10))

In [ ]:
analyze_distribution(creator_counter_ooxml)

In [ ]:
analyze_distribution(last_modified_by_counter_ooxml)

In [ ]:
analyze_distribution(creator_counter_ole2)

In [ ]:
analyze_distribution(last_modified_by_counter_ole2)

In [ ]:
# Selección de valores por frecuencia
N_creator_ooxml = 9
N_last_modified_by_ooxml = 11
N_creator_ole2 = 11
N_last_modified_by_ole2 = 13

filtered_creator_ooxml = {k: v for k, v in creator_counter_ooxml.items() if v <= N_creator_ooxml}
filtered_last_modified_by_ooxml = {k: v for k, v in last_modified_by_counter_ooxml.items() if v <= N_last_modified_by_ooxml}
filtered_creator_ole2 = {k: v for k, v in creator_counter_ole2.items() if v <= N_creator_ole2}
filtered_last_modified_by_ole2 = {k: v for k, v in last_modified_by_counter_ole2.items() if v <= N_last_modified_by_ole2}

In [ ]:
import networkx as nx
import math
from itertools import combinations, product

# ── helpers ──────────────────────────────────────────────────────────────────

def build_filtered(d, counter, N):
    """Devuelve {autor: [doc, ...]} filtrando por frecuencia <= N."""
    return {k: list(v) for k, v in d.items() if counter[k] <= N}

def all_docs_in_sources(sources):
    docs = set()
    for fields in sources.values():
        for author_docs in fields.values():
            for docs_list in author_docs.values():
                docs.update(docs_list)
    return docs

def global_freq(sources, author, field):
    """Nº total de docs asociados a un autor en un campo, entre todos los formatos."""
    total = 0
    for fmt in sources.values():
        if author in fmt.get(field, {}):
            total += len(fmt[field][author])
    return total

def idf(n_total, freq):
    """IDF estándar. Si freq==0 devuelve 0 para evitar división por cero."""
    return math.log(n_total / freq) if freq > 0 else 0

def add_edge(G, d1, d2, weight, relation, author):
    """
    Acumula peso en la arista y guarda toda la evidencia.
    Usa frozenset como clave para no duplicar en grafos no dirigidos.
    """
    if G.has_edge(d1, d2):
        e = G[d1][d2]
        e["weight"]    += weight
        e["n_links"]   += 1
        e["evidence"].append({"relation": relation, "author": author, "idf": round(weight, 4)})
    else:
        G.add_edge(d1, d2,
            weight   = weight,
            n_links  = 1,
            evidence = [{"relation": relation, "author": author, "idf": round(weight, 4)}]
        )

# ── construcción de fuentes ───────────────────────────────────────────────────

sources = {
    "ooxml": {
        "creator":          build_filtered(creator_dict_ooxml,          creator_counter_ooxml,          N_creator_ooxml),
        "last_modified_by": build_filtered(last_modified_by_dict_ooxml, last_modified_by_counter_ooxml, N_last_modified_by_ooxml),
    },
    "ole2": {
        "creator":          build_filtered(creator_dict_ole2,           creator_counter_ole2,           N_creator_ole2),
        "last_modified_by": build_filtered(last_modified_by_dict_ole2,  last_modified_by_counter_ole2,  N_last_modified_by_ole2),
    },
}

N_total = len(all_docs_in_sources(sources))

# ── construcción del grafo ────────────────────────────────────────────────────

G_meta = nx.Graph()

# 1. Mismo campo, mismo formato  (creator-creator   |   lmb-lmb)
for fmt, fields in sources.items():
    for field, author_docs in fields.items():
        for author, docs in author_docs.items():
            w = idf(N_total, global_freq(sources, author, field))
            for d1, d2 in combinations(docs, 2):
                add_edge(G_meta, d1, d2, w, f"{field}:{fmt}", author)

# 2. Cross-field: creator de fmt1 coincide con last_modified_by de fmt2
#    (cubre within-format Y cross-format en una sola pasada)
for fmt1, fmt2 in [("ooxml","ooxml"), ("ole2","ole2"), ("ooxml","ole2"), ("ole2","ooxml")]:
    for author, docs1 in sources[fmt1]["creator"].items():
        if author not in sources[fmt2]["last_modified_by"]:
            continue
        docs2 = sources[fmt2]["last_modified_by"][author]
        freq  = global_freq(sources, author, "creator") + global_freq(sources, author, "last_modified_by")
        w     = idf(N_total, freq)
        for d1, d2 in product(docs1, docs2):
            if d1 != d2:
                add_edge(G_meta, d1, d2, w, f"creator({fmt1})/lmb({fmt2})", author)

# 3. Mismo campo, cross-formato  (creator ooxml == creator ole2, idem lmb)
for field in ["creator", "last_modified_by"]:
    for author, docs_ooxml in sources["ooxml"][field].items():
        if author not in sources["ole2"][field]:
            continue
        docs_ole2 = sources["ole2"][field][author]
        w = idf(N_total, global_freq(sources, author, field))
        for d1, d2 in product(docs_ooxml, docs_ole2):
            add_edge(G_meta, d1, d2, w, f"{field}:cross-format", author)

# ── utilidades de consulta ────────────────────────────────────────────────────

def strong_edges(G, min_weight=1.0, min_links=1):
    """Filtra aristas por peso mínimo y/o número mínimo de vínculos."""
    return [
        (u, v, d) for u, v, d in G.edges(data=True)
        if d["weight"] >= min_weight and d["n_links"] >= min_links
    ]

def explain_edge(G, d1, d2):
    """Muestra por qué están conectados dos documentos."""
    if not G.has_edge(d1, d2):
        print("Sin conexión directa.")
        return
    e = G[d1][d2]
    print(f"Peso total : {e['weight']:.4f}")
    print(f"Nº vínculos: {e['n_links']}")
    for ev in e["evidence"]:
        print(f"  · {ev['relation']}  autor={ev['author']}  idf={ev['idf']}")

In [ ]:
# iteramos por todos los nodos para coger su hash, y poner como atributo el formato del documento usando check_ooxml y check_ole2
# tambien miramos con su hash en csv_full la columna "Normalized_Tag" que es el apt
for node in G_meta.nodes():
    path = f"/home/jupyter/binaries/{node}"
    if os.path.exists(path):
        if check_ooxml(path):
            G_meta.nodes[node]['format'] = 'ooxml'
        elif check_ole2(path):
            G_meta.nodes[node]['format'] = 'ole2'
        else:
            G_meta.nodes[node]['format'] = 'unknown'
    else:
        G_meta.nodes[node]['format'] = 'missing'

    # buscar en csv_full la columna Normalized_Tag
    tag_row = csv_full[csv_full['sha256'] == node]
    if not tag_row.empty:
        G_meta.nodes[node]['Normalized_Tag'] = tag_row.iloc[0]['Normalized_Tag']
    else:
        G_meta.nodes[node]['Normalized_Tag'] = None

In [ ]:
# exportar el grafo gephi
nx.write_gexf(G_meta, "meta_graph.gexf")

# Enriqueimiento y metricas

Limpieza del grafo, detección de comunidades y evaluación de las agrupaciones.

## metricas 

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from communities.algorithms import louvain_method
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, adjusted_rand_score

# ── CONFIGURACIÓN ──────────────────────────────────────────
GRAPH_PATH  = "/home/jupyter/workspace/graph_phash.gexf"   # ruta a tu archivo
APT_ATTR    = "apt"             # nombre del atributo de nodo con la etiqueta APT
PURITY_THR  = 0.7              # umbral para considerar un clúster "puro"
# ───────────────────────────────────────────────────────────


# abrir el grafo y convertirlo 
G_meta = nx.read_gexf(GRAPH_PATH)

In [ ]:
# borrar las aristas que unen al mismo documento (self-loops)
G_meta.remove_edges_from(nx.selfloop_edges(G_meta))

In [ ]:
# borrar los nodos aislados
isolated_meta = list(nx.isolates(G_meta))
G_meta.remove_nodes_from(isolated_meta)

In [ ]:
#exportar el grafo limpio a un nuevo archivo
nx.write_gexf(G_meta, "meta_graph.gexf")

In [ ]:
# ver sobre el csv_full cuantos no hay en el grafo
hashes_in_graph = set(G_meta.nodes())
hashes_in_csv = set(csv_full['sha256'])
missing_hashes = hashes_in_csv - hashes_in_graph
print(f"Documentos en CSV pero no en grafo: {len(missing_hashes)}")

# ver cunatos nodos no tienen conexión con otros nodos (grado 0)
isolated_nodes = [n for n, d in G_meta.degree() if d == 0]
print(f"Nodos aislados (grado 0): {len(isolated_nodes)}")

#numero de nodos y aristas
print(f"Nodos: {G_meta.number_of_nodes()}")
print(f"Aristas: {G_meta.number_of_edges()}")

# densidad, numero de componentes conexas, tamaño de la mayor componente
print(f"Densidad: {nx.density(G_meta)*100:.2f}% | Aristas posibles: {G_meta.number_of_nodes() * (G_meta.number_of_nodes() - 1) / 2} | Aristas reales: {G_meta.number_of_edges()}")
components = list(nx.connected_components(G_meta))
print(f"Componentes conexas: {len(components)}")
largest_component_size = len(max(components, key=len))
print(f"Tamaño de la mayor componente: {largest_component_size}")

#coef de clustering medio, grado medio, grado mediano, grado máximo
clustering_coeffs = list(nx.clustering(G_meta, weight="weight").values())
degrees = [d for _, d in G_meta.degree()]
print(f"Coeficiente de clustering medio: {np.mean(clustering_coeffs):.4f}")
print(f"Grado medio: {np.mean(degrees):.2f} | Grado mediano: {np.median(degrees)} | Grado máximo: {max(degrees)}")

In [ ]:
# numero de comunidades detectadas
communities = list(nx.community.greedy_modularity_communities(G_meta))
print(f"Comunidades detectadas: {len(communities)}")

#modularidad
modularity = nx.community.modularity(G_meta, communities)
print(f"Modularidad: {modularity:.4f}")

# aplicar gini coefficient a cada comunidad para ver la desigualdad de tamaño entre comunidades
community_sizes = [len(c) for c in communities]
def gini_coefficient(x):
    """Compute Gini coefficient of array of values"""
    x = np.array(x, dtype=float)
    if np.amin(x) < 0:
        x -= np.amin(x)  # Values cannot be negative
    x += 0.0000001  # Values cannot be 0
    x = np.sort(x)
    index = np.arange(1, x.shape[0] + 1)
    n = x.shape[0]
    return ((np.sum((2 * index - n - 1) * x)) / (n * np.sum(x)))

gini = gini_coefficient(community_sizes)
print(f"Gini coefficient de tamaños de comunidades: {gini:.4f} (0 = todos iguales, 1 = un hub gigante)")

In [ ]:
# Nodos con el mismo APT por comunidad
# mostrar a parte del apt dominante, los distintos APTs que hay en la comunidad y su cobertura
for i, community in enumerate(communities):
    apt_counter = Counter()
    for node in community:
        apt = G_meta.nodes[node].get("apt", None)
        if apt:
            apt_counter[apt] += 1
    total_nodes = len(community)
    if total_nodes == 0:
        continue
    dominant_apt, dominant_count = apt_counter.most_common(1)[0] if apt_counter else (None, 0)
    coverage = dominant_count / total_nodes * 100 if total_nodes > 0 else 0
    print(f"Comunidad {i}: {total_nodes} nodos | APT dominante: {dominant_apt} ({dominant_count} nodos, {coverage:.2f}%) | APTs distintos: {len(apt_counter)}")
    
    # Desglose de todos los APTs en la comunidad
    for apt, count in apt_counter.most_common():
        pct = count / total_nodes * 100
        bar = "█" * int(pct / 5)
        print(f"    {apt:<20} {count:>3} nodos  {pct:>5.1f}%  {bar}")
    print()

In [ ]:
# Construir mapa nodo -> comunidad
node_to_comm = {}
for i, community in enumerate(communities):
    for node in community:
        node_to_comm[node] = i

# Agrupar por APT
apt_to_comms = {}
for node in G_meta.nodes():
    apt = G_meta.nodes[node].get("apt", None)
    if not apt:
        continue
    comm = node_to_comm.get(node)
    if apt not in apt_to_comms:
        apt_to_comms[apt] = Counter()
    apt_to_comms[apt][comm] += 1

# Mostrar por APT
for apt, comm_counter in sorted(apt_to_comms.items()):
    total = sum(comm_counter.values())
    dominant_comm, dominant_count = comm_counter.most_common(1)[0]
    coverage = dominant_count / total * 100
    print(f"{apt:<20} {total} docs | comunidad dominante: {dominant_comm} ({dominant_count} docs, {coverage:.1f}%) | en {len(comm_counter)} comunidades")
    
    for comm_id, count in comm_counter.most_common():
        pct = count / total * 100
        bar = "█" * int(pct / 5)
        print(f"    Comunidad {comm_id:<4} {count:>3} docs  {pct:>5.1f}%  {bar}")
    print()

In [ ]:
# Importar las métricas de sklearn y pandas para la tabla
from sklearn.metrics import homogeneity_completeness_v_measure
import pandas as pd

# 1. Preparar las listas de Ground Truth (y_true) y Predicción (y_pred)
y_true = []
y_pred = []

# Iteramos sobre los nodos del grafo para extraer su APT real y su comunidad asignada
for node in G_meta.nodes():
    apt = G_meta.nodes[node].get("apt", None)    # <--- aqui ir cambiando entre apt y Normalized_Tag
    comm_id = node_to_comm.get(node)
    
    # Nodos con una etiqueta APT válida
    if apt and pd.notna(apt) and comm_id is not None:
        y_true.append(apt)
        y_pred.append(comm_id)

# 2. Calcular las métricas
homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y_true, y_pred)

# 3. Mostrar los resultados
print("="*60)
print("MÉTRICAS DE EVALUACIÓN DE CLUSTERING (GROUND TRUTH)")
print("="*60)
print(f"Homogeneidad (¿Son puros los clústeres?):          {homogeneity:.4f} (Rango: 0-1)")
print(f"Completitud  (¿Está cada APT en un solo clúster?): {completeness:.4f} (Rango: 0-1)")
print(f"V-Measure    (Score global / Media armónica):      {v_measure:.4f} (Rango: 0-1)")
print("="*60)

# 4. BONUS: Crear una matriz de contingencia (Cross-Tabulation)
# Esto genera una tabla donde las filas son los APTs y las columnas son tus comunidades
df_eval = pd.DataFrame({'APT (Real)': y_true, 'Comunidad (Predicha)': y_pred})
contingency_matrix = pd.crosstab(df_eval['APT (Real)'], df_eval['Comunidad (Predicha)'])

print("\nMatriz de Contingencia (APT vs Comunidades):")
# Mostrar la tabla estilizada (en Jupyter se verá como un DataFrame bonito)
display(contingency_matrix)

In [ ]:
# ver cuantos nodos y de que apt se ha quedado fuera de todas las comunidades (no tienen comunidad asignada)
nodos_sin_comunidad = [n for n in G_meta.nodes() if n not in node_to_comm]
print(f"Nodos sin comunidad asignada: {len(nodos_sin_comunidad)}")
apt_counter_sin_comunidad = Counter()
for node in nodos_sin_comunidad:
    apt = G_meta.nodes[node].get("apt", None)
    if apt:
        apt_counter_sin_comunidad[apt] += 1
print("Distribución de APTs en nodos sin comunidad:")
for apt, count in apt_counter_sin_comunidad.most_common():
    print(f"  {apt:<20} {count:>3} nodos")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Apuntamos a la parte limpia de tu diccionario
apt_db = apt_synonyms['apt_alias_merged']

# 2. Función de búsqueda adaptada a tu nueva estructura (synonyms y nation)
def get_apt_context(apt_name, db):
    for group_key, info in db.items():
        # Pasamos a minúsculas para evitar fallos por mayúsculas/minúsculas
        synonyms = {s.lower().strip() for s in info.get("synonyms", [])}
        
        if apt_name.lower().strip() == group_key.lower().strip() or apt_name.lower().strip() in synonyms:
            nations = info.get("nation", [])
            
            # Buscar el primer país válido que no sea "Unknown" u "Others"
            for n in nations:
                n_low = n.lower().strip()
                if n_low not in ["others", "unknown", "worldwide", "financially_motivated", "multiple", "private_sector_offensive_actor", "set()"]:
                    return group_key, n_low
            return group_key, "unknown"
            
    return "Unknown Group", "unknown"

# 3. Mapeo de países a Paletas de Color (ampliado para cubrir tu nuevo dict)
country_palettes = {
    "north korea": plt.cm.Reds,
    "south korea": plt.cm.Reds,
    "russia": plt.cm.Blues,
    "belarus": plt.cm.Blues,
    "ukraine": plt.cm.Blues,
    "china": plt.cm.Greens,
    "iran": plt.cm.Oranges,
    "pakistan": plt.cm.Purples,
    "india": plt.cm.Purples,
    "middle east": plt.cm.YlOrBr,
    "syria": plt.cm.YlOrBr,
    "lebanon": plt.cm.YlOrBr,
    "palestina": plt.cm.YlOrBr,
    "gaza": plt.cm.YlOrBr,
    "vietnam": plt.cm.GnBu,
    "turkey": plt.cm.pink,
    "united states": plt.cm.copper,
    "united states of america": plt.cm.copper,
    "brazil": plt.cm.YlGn,
}
default_palette = plt.cm.Greys

# 4. Generar la jerarquía de los APTs presentes en tu matriz de contingencia
apts_in_matrix = contingency_matrix.index.tolist()
apt_hierarchy = {}

for apt in apts_in_matrix:
    group, country = get_apt_context(apt, apt_db)
    if country not in apt_hierarchy:
        apt_hierarchy[country] = {}
    if group not in apt_hierarchy[country]:
        apt_hierarchy[country][group] = []
    apt_hierarchy[country][group].append(apt)

# 5. Asignar un color HEX/RGBA a cada APT basado en su país y grupo
apt_colors = {}
for country, groups in apt_hierarchy.items():
    palette = country_palettes.get(country, default_palette)
    total_apts_in_country = sum(len(apts) for apts in groups.values())
    
    # Generar tonos espaciados (del 0.35 al 0.85 para que no sea ni blanco puro ni negro puro)
    if total_apts_in_country > 1:
        shades = np.linspace(0.35, 0.85, total_apts_in_country)
    else:
        shades = [0.6]
        
    shade_idx = 0
    for group, apts in groups.items():
        for apt in apts:
            apt_colors[apt] = palette(shades[shade_idx])
            shade_idx += 1

# Extraer la lista de colores en el mismo orden en el que están las columnas de la gráfica
colors_for_plot = [apt_colors[apt] for apt in contingency_matrix.T.columns]

# GRÁFICO 1: Pureza de las Comunidades (Homogeneidad al 100%)
# Normalizamos para que todas las barras lleguen al 100%
contingency_matrix_norm = contingency_matrix.T.div(contingency_matrix.T.sum(axis=1), axis=0) * 100

ax1 = contingency_matrix_norm.plot(
    kind='bar', 
    stacked=True, 
    figsize=(18, 7), 
    width=0.65,
    color=colors_for_plot,
    edgecolor='black',
    linewidth=0.5
)

plt.title('Pureza Semántica de los Clústeres (Porcentaje de Composición)', fontsize=16, pad=15)
plt.xlabel('ID de la Comunidad Predicha', fontsize=12)
plt.ylabel('Porcentaje de Documentos (%)', fontsize=12)
plt.legend(title='APT (Color por País/Grupo)', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9, ncol=2)
ax1.tick_params(axis='x', labelrotation=45)
for label in ax1.get_xticklabels():
    label.set_ha('right')
plt.tight_layout()
plt.savefig('grafico_pureza_semantico.png', dpi=300)
plt.show()

# GRÁFICO 2: Composición Absoluta con Colores Semánticos
ax2 = contingency_matrix.T.plot(
    kind='bar', 
    stacked=True, 
    figsize=(18, 7), 
    width=0.65,
    color=colors_for_plot, 
    edgecolor='black',
    linewidth=0.5
)

plt.title('Composición Absoluta de APTs por Comunidad Descubierta PHASH', fontsize=16, pad=15)
plt.xlabel('ID de la Comunidad Predicha', fontsize=12)
plt.ylabel('Número de Documentos', fontsize=12)
plt.legend(title='APT (Color por País/Grupo)', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9, ncol=2)
ax2.tick_params(axis='x', labelrotation=45)
for label in ax2.get_xticklabels():
    label.set_ha('right')
plt.tight_layout()
plt.savefig('grafico_absoluto_semantico.png', dpi=300)
plt.show()

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics import homogeneity_completeness_v_measure

# 1. Función de Gini
def gini_coefficient(x):
    """Calcula el coeficiente de Gini de un array de valores (0 = igualdad, 1 = máxima desigualdad)"""
    x = np.array(x, dtype=float)
    if len(x) == 0: return 0
    if np.amin(x) < 0:
        x -= np.amin(x) # Los valores no pueden ser negativos
    x += 0.0000001      # Evitar división por cero
    x = np.sort(x)
    index = np.arange(1, x.shape[0] + 1)
    n = x.shape[0]
    return ((np.sum((2 * index - n - 1) * x)) / (n * np.sum(x)))

# 2. Evaluación del grafo y sus métricas
def evaluar_grafo(G, nombre_grafo="Grafo"):
    n_nodos = G.number_of_nodes()
    n_aristas = G.number_of_edges()
    densidad = nx.density(G) * 100
    comp_conexas = nx.number_connected_components(G)
    
    # Coeficiente de Clustering Medio
    clustering_coeffs = list(nx.clustering(G, weight="weight").values())
    avg_clustering = np.mean(clustering_coeffs) if clustering_coeffs else 0
    
    # Algoritmo greedy de modularidad
    communities = list(nx.community.greedy_modularity_communities(G))
    modularidad = nx.community.modularity(G, communities)
    
    # Tamaños de las comunidades y Gini
    community_sizes = [len(c) for c in communities]
    gini_comunidades = gini_coefficient(community_sizes)
    
    y_true = []
    y_pred = []
    
    # Mapeo rápido de nodo -> id_comunidad
    node_to_comm = {}
    for i, comm in enumerate(communities):
        for node in comm:
            node_to_comm[node] = i
            
    for node in G.nodes():
        apt = G.nodes[node].get("apt", None)
        comm_id = node_to_comm.get(node)
        
        # Nodos con un APT conocido
        if apt and pd.notna(apt) and comm_id is not None:
            y_true.append(apt)
            y_pred.append(comm_id)
            
    # if y_true:
    #     homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y_true, y_pred)
    # else:
    #     homogeneity, completeness, v_measure = 0, 0, 0
        
    return {
        "Grafo": nombre_grafo,
        "Nodos": n_nodos,
        "Aristas": n_aristas,
        "Densidad (%)": round(densidad, 2),
        "Comp. Conexas": comp_conexas,
        "Clustering Medio": round(avg_clustering, 4),
        "Modularidad": round(modularidad, 4),
        #"Gini (Comunidades)": round(gini_comunidades, 4),
        "Homogeneidad": round(homogeneity, 4),
        "Completitud": round(completeness, 4),
        "V-Measure": round(v_measure, 4)
    }


resultados = []

# Evaluar cada uno y pasar las variables reales
resultados.append(evaluar_grafo(G_meta, "PHASH"))
# resultados.append(evaluar_grafo(G_rsid, "RSID"))
# resultados.append(evaluar_grafo(G_clsid, "CLSIDs"))
# resultados.append(evaluar_grafo(G_fusion, "Grafo Fusionado"))


df_resultados = pd.DataFrame(resultados)
display(df_resultados)

# Para exportar a LaTeX directo 
# print(df_resultados.to_latex(index=False))